In [1]:
# Cell 1: Imports and setup
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import requests
import io
from kloppy import impect
from kloppy.utils import github_resolve_raw_data_url
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful")



All imports successful


In [2]:
# Cell 2: Setup project directories
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")

# Create figures directory if doesn't exist
FIGURES_DIR.mkdir(exist_ok=True)

print(f" Paths configured:")
print(f"   Data: {DATA_DIR}")
print(f"   Processed: {PROCESSED_DIR}")
print(f"   Figures: {FIGURES_DIR}")

# List what processed files we have
print(f"\n Available processed files:")
import os
for file in sorted(os.listdir(PROCESSED_DIR)):
    if file.endswith('.parquet'):
        size_mb = os.path.getsize(PROCESSED_DIR / file) / 1_000_000
        print(f"   {file:45s} ({size_mb:>5.1f} MB)")

 Paths configured:
   Data: ../data
   Processed: ../data/processed
   Figures: ../figures

 Available processed files:
   all_events_with_points.parquet                ( 17.0 MB)
   all_matches_with_zones.parquet                ( 15.5 MB)
   matches_metadata.parquet                      (  0.0 MB)
   player_metadata.parquet                       (  0.0 MB)
   player_names_complete.parquet                 (  0.0 MB)
   player_ratings_FINAL_with_market_values.parquet (  0.1 MB)
   player_ratings_with_8_metrics.parquet         (  0.1 MB)
   squads_metadata.parquet                       (  0.0 MB)


In [3]:
# Cell 3: Load base event data (with zones only)

print("\nLoading base event data...")


# Load events with zones (from notebook 01)
all_events = pl.read_parquet(PROCESSED_DIR / "all_matches_with_zones.parquet")

print(f" Loaded {len(all_events):,} events")
print(f"   Unique players: {all_events['player_id'].n_unique()}")
print(f"   Unique matches: {all_events['match_id'].n_unique()}")

# Show what columns we have
print(f"\n Available columns:")
print(all_events.columns)

# Show sample
print(f"\n Sample events:")
print(all_events.select([
    'event_type', 'player_id', 'zone', 'result', 'success',
    'coordinates_x', 'end_coordinates_x'
]).sample(5))


Loading base event data...
 Loaded 962,990 events
   Unique players: 494
   Unique matches: 306

 Available columns:
['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp', 'ball_state', 'ball_owning_team', 'team_id', 'player_id', 'coordinates_x', 'coordinates_y', 'end_coordinates_x', 'end_coordinates_y', 'receiver_player_id', 'body_part_type', 'set_piece_type', 'result', 'success', 'duel_type', 'is_under_pressure', 'pass_type', 'goalkeeper_type', 'match_id', 'zone', 'end_zone', 'card_type']

 Sample events:
shape: (5, 7)
┌───────────────┬───────────┬───────────────┬────────────┬─────────┬───────────────┬───────────────┐
│ event_type    ┆ player_id ┆ zone          ┆ result     ┆ success ┆ coordinates_x ┆ end_coordinat │
│ ---           ┆ ---       ┆ ---           ┆ ---        ┆ ---     ┆ ---           ┆ es_x          │
│ str           ┆ str       ┆ str           ┆ str        ┆ bool    ┆ f64           ┆ ---           │
│               ┆           ┆               ┆        

In [4]:
all_events

event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,coordinates_y,end_coordinates_x,end_coordinates_y,receiver_player_id,body_part_type,set_piece_type,result,success,duel_type,is_under_pressure,pass_type,goalkeeper_type,match_id,zone,end_zone,card_type
str,str,i64,duration[μs],duration[μs],str,str,str,str,f64,f64,f64,f64,str,str,str,str,bool,str,bool,str,str,i32,str,str,str
"""4858179098""","""PASS""",1,0µs,332ms,"""alive""","""33""","""33""","""204""",0.0,0.0,null,null,null,"""RIGHT_FOOT""","""KICK_OFF""","""INCOMPLETE""",false,null,null,null,null,122838,"""middle_third""",null,null
"""4858179099""","""GENERIC:NO_VIDEO""",1,332ms,null,"""alive""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,122838,null,null,null
"""4858179100""","""GENERIC:RECEPTION""",1,1s 651ms,null,"""alive""","""33""","""33""","""1202""",-31.1,4.7,null,null,null,null,null,null,null,null,null,null,null,122838,"""defensive_third""",null,null
"""4858179101""","""CARRY""",1,1s 651100µs,4s 193ms,"""alive""","""33""","""33""","""1202""",-31.1,4.7,-30.9,-0.1,null,"""RIGHT_FOOT""",null,"""COMPLETE""",true,null,null,null,null,122838,"""defensive_third""","""defensive_third""",null
"""4858179102""","""PASS""",1,4s 192999µs,6s 904999µs,"""alive""","""33""","""33""","""1202""",-30.9,-0.1,13.4,27.8,null,"""RIGHT_FOOT""",null,"""INCOMPLETE""",false,null,null,null,null,122838,"""defensive_third""","""middle_third""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""4607694199""","""INTERCEPTION""",2,44m 47s 878999µs,null,"""alive""","""29""","""36""","""1415""",-16.9,17.2,null,null,null,"""RIGHT_FOOT""",null,"""SUCCESS""",true,null,true,null,null,123143,"""middle_third""",null,null
"""4607694200""","""CARRY""",2,44m 47s 879100µs,44m 48s 184ms,"""alive""","""29""","""36""","""1415""",-16.9,17.2,-16.9,18.1,null,"""RIGHT_FOOT""",null,"""COMPLETE""",true,null,true,null,null,123143,"""middle_third""","""middle_third""",null
"""4607694201""","""CLEARANCE""",2,44m 48s 184ms,null,"""alive""","""29""","""36""","""1415""",-16.9,18.1,null,null,null,"""RIGHT_FOOT""",null,null,null,null,true,null,null,123143,"""middle_third""",null,null


In [5]:
# Cell 4: Helper functions to identify special event types

print("DEFINING HELPER FUNCTIONS")
print("="*70)

# =============================================================================
# UTILITY FUNCTION
# =============================================================================

def calculate_distance(start_x, start_y, end_x, end_y):
    """
    Calculate Euclidean distance between two points.
    
    Args:
        start_x, start_y: Starting coordinates
        end_x, end_y: Ending coordinates
    
    Returns:
        float: Distance in meters, or None if coordinates missing
    """
    if None in [start_x, start_y, end_x, end_y]:
        return None
    
    distance = np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2)
    return distance


# =============================================================================
# ATTACKING HELPERS
# =============================================================================

def is_progressive_pass(row):
    """
    Check if pass is progressive (advances ball ≥10m toward opponent goal).
    
    Logic: Progressive if (end_x - start_x) ≥ 10 meters
    Threshold: 10m is industry standard (StatsBomb, Opta)
    
    Returns:
        bool: True if progressive, False otherwise
    """
    start_x = row.get('coordinates_x')
    end_x = row.get('end_coordinates_x')
    
    if start_x is None or end_x is None:
        return False
    
    forward_distance = end_x - start_x
    return forward_distance >= 10.0


def is_progressive_carry(row):
    """
    Check if carry/dribble is progressive (advances ball ≥10m forward).
    
    Logic: Same as progressive pass, applied to carries
    
    Returns:
        bool: True if progressive, False otherwise
    """
    start_x = row.get('coordinates_x')
    end_x = row.get('end_coordinates_x')
    
    if start_x is None or end_x is None:
        return False
    
    forward_distance = end_x - start_x
    return forward_distance >= 10.0


def is_long_pass(row):
    """
    Check if pass is long-range (total distance ≥30m).
    
    Logic: Calculate Euclidean distance using Pythagorean theorem
    Threshold: 30m ≈ switching play or long ball
    
    Returns:
        bool: True if long pass, False otherwise
    """
    start_x = row.get('coordinates_x')
    start_y = row.get('coordinates_y')
    end_x = row.get('end_coordinates_x')
    end_y = row.get('end_coordinates_y')
    
    distance = calculate_distance(start_x, start_y, end_x, end_y)
    
    if distance is None:
        return False
    
    return distance >= 30.0


def is_into_penalty_area(row):
    """
    Check if pass/carry ends in opponent's penalty area.
    
    Logic: Penalty area in secondspectrum coordinates:
           - x > 35.5 (near opponent goal, 16.5m from goal line)
           - -9.15 < y < 9.15 (width of penalty box, ±10m from center)
    
    Returns:
        bool: True if ends in penalty area, False otherwise
    """
    end_x = row.get('end_coordinates_x')
    end_y = row.get('end_coordinates_y')
    
    if end_x is None or end_y is None:
        return False
    
    in_box_x = end_x > 35.5
    in_box_y = -9.15 < end_y < 9.15
    
    return in_box_x and in_box_y


def is_into_final_third(row):
    """
    Check if pass/carry crosses into attacking third.
    
    Logic: Must START outside attacking third and END inside it
    
    Returns:
        bool: True if crosses into final third, False otherwise
    """
    start_zone = row.get('zone')
    end_zone = row.get('end_zone')
    
    if start_zone is None or end_zone is None:
        return False
    
    # Must start outside attacking third and end inside it
    crosses_into_attack = (
        start_zone != 'attacking_third' and 
        end_zone == 'attacking_third'
    )
    
    return crosses_into_attack


def is_carry_into_box(row):
    """
    Check if carry/dribble ends in opponent's penalty area.
    
    Logic: Same as is_into_penalty_area, but for carries
    High value: beating defenders with the ball into dangerous area
    
    Returns:
        bool: True if dribble into box, False otherwise
    """
    # Reuse penalty area logic
    return is_into_penalty_area(row)


# =============================================================================
# DEFENSIVE HELPERS
# =============================================================================

def is_in_own_penalty_area(row):
    """
    Check if event occurred in own penalty area (critical defensive zone).
    
    Logic: Own penalty area in secondspectrum:
           - x < -35.5 (near own goal)
           - -9.15 < y < 9.15 (width of box)
    
    Returns:
        bool: True if in own penalty area, False otherwise
    """
    x = row.get('coordinates_x')
    y = row.get('coordinates_y')
    
    if x is None or y is None:
        return False
    
    in_own_box_x = x < -35.5
    in_box_y = -9.15 < y < 9.15
    
    return in_own_box_x and in_box_y


# =============================================================================
# TEST HELPER FUNCTIONS
# =============================================================================

print("\n Helper functions defined:")
print("   1. calculate_distance() - Euclidean distance")
print("   2. is_progressive_pass() - Forward ≥10m")
print("   3. is_progressive_carry() - Forward ≥10m")
print("   4. is_long_pass() - Total distance ≥30m")
print("   5. is_into_penalty_area() - Ends in opponent box")
print("   6. is_into_final_third() - Crosses into attacking third")
print("   7. is_carry_into_box() - Dribble into box")
print("   8. is_in_own_penalty_area() - In own box (critical defense)")

# Test functions on sample events
print("\n" + "="*70)
print("TESTING HELPER FUNCTIONS")
print("="*70)

test_events = all_events.filter(
    pl.col('event_type').is_in(['PASS', 'CARRY'])
).sample(10)

print("\nSample event tests:")
for i, row_dict in enumerate(test_events.to_dicts()[:5], 1):
    print(f"\nEvent {i}: {row_dict['event_type']}")
    print(f"  Coordinates: ({row_dict.get('coordinates_x', 'N/A'):.1f}, {row_dict.get('coordinates_y', 'N/A'):.1f}) "
          f"→ ({row_dict.get('end_coordinates_x', 'N/A'):.1f}, {row_dict.get('end_coordinates_y', 'N/A'):.1f})")
    print(f"  Zone: {row_dict.get('zone')} → {row_dict.get('end_zone')}")
    print(f"  Progressive: {is_progressive_pass(row_dict) if row_dict['event_type'] == 'PASS' else is_progressive_carry(row_dict)}")
    print(f"  Long: {is_long_pass(row_dict)}")
    print(f"  Into box: {is_into_penalty_area(row_dict)}")
    print(f"  Into final third: {is_into_final_third(row_dict)}")

print("\n All helper functions working correctly")

DEFINING HELPER FUNCTIONS

 Helper functions defined:
   1. calculate_distance() - Euclidean distance
   2. is_progressive_pass() - Forward ≥10m
   3. is_progressive_carry() - Forward ≥10m
   4. is_long_pass() - Total distance ≥30m
   5. is_into_penalty_area() - Ends in opponent box
   6. is_into_final_third() - Crosses into attacking third
   7. is_carry_into_box() - Dribble into box
   8. is_in_own_penalty_area() - In own box (critical defense)

TESTING HELPER FUNCTIONS

Sample event tests:

Event 1: CARRY
  Coordinates: (9.0, 9.2) → (7.2, -3.7)
  Zone: middle_third → middle_third
  Progressive: False
  Long: False
  Into box: False
  Into final third: False

Event 2: CARRY
  Coordinates: (6.3, -20.4) → (6.0, -20.4)
  Zone: middle_third → middle_third
  Progressive: False
  Long: False
  Into box: False
  Into final third: False

Event 3: CARRY
  Coordinates: (42.5, 9.5) → (43.3, 9.0)
  Zone: attacking_third → attacking_third
  Progressive: False
  Long: False
  Into box: True
  Into

In [6]:
# Cell 5: Calculate metric points for all events using helper functions

print("\nCALCULATING METRIC POINTS FOR ALL EVENTS")
print("="*70)
print(f"Processing {len(all_events):,} events...")
print(" This will take ~30-45 seconds...")

import time
start_time = time.time()

# Convert to list of dicts for faster iteration
events_list = all_events.to_dicts()

# Initialize point lists
finishing_points = []
chance_creation_points = []
ball_progression_points = []
dribbling_points = []
ball_winning_points = []
defensive_actions_points = []
passing_accuracy_points = []
long_passing_points = []

# =============================================================================
# POINT CALCULATION FUNCTIONS
# =============================================================================

def calc_finishing_points(row):
    """Calculate finishing metric points"""
    if row['event_type'] != 'SHOT':
        return 0.0
    
    result = row.get('result')
    if result == 'GOAL':
        return 10.0
    elif result == 'SAVED':
        return 2.0
    elif result == 'BLOCKED':
        return 0.5
    elif result == 'OFF_TARGET':
        return 0.3
    return 0.0


def calc_chance_creation_points(row):
    """Calculate chance creation metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    pass_type = row.get('pass_type')
    
    # Assist (pass leading to goal)
    if pass_type == 'SHOT_ASSIST':
        return 10.0
    
    # Pass into penalty area
    if row.get('result') == 'COMPLETE' and is_into_penalty_area(row):
        return 3.0
    
    return 0.0


def calc_ball_progression_points(row):
    """Calculate ball progression metric points"""
    points = 0.0
    
    # Progressive pass
    if row['event_type'] == 'PASS' and row.get('result') == 'COMPLETE':
        if is_progressive_pass(row):
            points += 2.0
        elif is_into_final_third(row):
            points += 1.5 #check
    
    # Progressive carry
    elif row['event_type'] == 'CARRY' and row.get('result') == 'COMPLETE':
        if is_progressive_carry(row):
            points += 2.5
    
    return points


def calc_dribbling_points(row):
    """Calculate dribbling metric points"""
    points = 0.0
    
    # Carry into penalty box (high value)
    if row['event_type'] == 'CARRY' and row.get('result') == 'COMPLETE':
        if is_carry_into_box(row):
            points += 3.0
        elif is_progressive_carry(row):
            points += 1.5
    
    # Duel won in attacking third (winning ball while attacking)
    elif row['event_type'] == 'DUEL' and row.get('result') == 'WON':
        if row.get('zone') == 'attacking_third':
            points += 1.0
    
    return points


def calc_ball_winning_points(row):
    """Calculate ball winning metric points"""
    points = 0.0
    zone = row.get('zone')
    
    # Duels won
    if row['event_type'] == 'DUEL' and row.get('result') == 'WON':
        if zone == 'defensive_third':
            points += 3.0
        elif zone == 'middle_third':
            points += 2.0
        elif zone == 'attacking_third':
            points += 1.0
    
    # Interceptions
    elif row['event_type'] == 'INTERCEPTION':
        if zone == 'defensive_third':
            points += 2.5
        elif zone == 'middle_third':
            points += 2.0
        elif zone == 'attacking_third':
            points += 1.5
    
    return points


def calc_defensive_actions_points(row):
    """Calculate defensive actions metric points"""
    points = 0.0
    zone = row.get('zone')
    
    # Clearances (removing danger)
    if row['event_type'] == 'CLEARANCE':
        # Extra points if in own penalty area (critical!)
        if is_in_own_penalty_area(row):
            points += 3.0
        elif zone == 'defensive_third':
            points += 2.0
        elif zone == 'middle_third':
            points += 1.0
    
    # Recoveries (regaining loose balls)
    elif row['event_type'] == 'RECOVERY':
        if zone == 'defensive_third':
            points += 1.5
        elif zone == 'middle_third':
            points += 1.0
        elif zone == 'attacking_third':
            points += 0.8
    
    return points


def calc_passing_accuracy_points(row):
    """Calculate passing accuracy metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    result = row.get('result')
    
    if result == 'COMPLETE':
        # Bonus if under pressure
        if row.get('is_under_pressure', False):
            return 1.0
        return 0.5
    elif result == 'INCOMPLETE':
        return -0.3
    
    return 0.0


def calc_long_passing_points(row):
    """Calculate long passing metric points"""
    if row['event_type'] != 'PASS':
        return 0.0
    
    if is_long_pass(row):
        if row.get('result') == 'COMPLETE':
            return 2.0
        elif row.get('result') == 'INCOMPLETE':
            return -0.5
    
    return 0.0


# =============================================================================
# CALCULATE POINTS FOR ALL EVENTS
# =============================================================================

for i, row in enumerate(events_list):
    finishing_points.append(calc_finishing_points(row))
    chance_creation_points.append(calc_chance_creation_points(row))
    ball_progression_points.append(calc_ball_progression_points(row))
    dribbling_points.append(calc_dribbling_points(row))
    ball_winning_points.append(calc_ball_winning_points(row))
    defensive_actions_points.append(calc_defensive_actions_points(row))
    passing_accuracy_points.append(calc_passing_accuracy_points(row))
    long_passing_points.append(calc_long_passing_points(row))
    
    # Progress indicator
    if (i + 1) % 200000 == 0:
        elapsed = time.time() - start_time
        progress = (i + 1) / len(events_list)
        remaining = (elapsed / progress) * (1 - progress)
        print(f"  Processed {i+1:,}/{len(events_list):,} ({progress:.1%}) | "
              f"Elapsed: {elapsed:.1f}s | Remaining: {remaining:.1f}s")

# Add as new columns
all_events = all_events.with_columns([
    pl.Series('finishing_points', finishing_points),
    pl.Series('chance_creation_points', chance_creation_points),
    pl.Series('ball_progression_points', ball_progression_points),
    pl.Series('dribbling_points', dribbling_points),
    pl.Series('ball_winning_points', ball_winning_points),
    pl.Series('defensive_actions_points', defensive_actions_points),
    pl.Series('passing_accuracy_points', passing_accuracy_points),
    pl.Series('long_passing_points', long_passing_points),
])

elapsed = time.time() - start_time
print(f"\n Calculated all metric points in {elapsed:.1f} seconds")

# Summary
print("\n" + "="*70)
print("METRIC POINTS SUMMARY")
print("="*70)

metrics = [
    'finishing_points', 'chance_creation_points', 'ball_progression_points',
    'dribbling_points', 'ball_winning_points', 'defensive_actions_points',
    'passing_accuracy_points', 'long_passing_points'
]

for metric in metrics:
    total = all_events[metric].sum()
    positive = (all_events[metric] > 0).sum()
    negative = (all_events[metric] < 0).sum()
    print(f"{metric:30s}: Total={total:>9,.0f} | Positive={positive:>7,} | Negative={negative:>6,}")

print("\n Sample events with calculated points:")
print(all_events.select([
    'event_type', 'zone', 'result',
    'finishing_points', 'chance_creation_points', 'ball_progression_points',
    'ball_winning_points'
]).sample(5))


CALCULATING METRIC POINTS FOR ALL EVENTS
Processing 962,990 events...
 This will take ~30-45 seconds...
  Processed 200,000/962,990 (20.8%) | Elapsed: 3.9s | Remaining: 14.8s
  Processed 400,000/962,990 (41.5%) | Elapsed: 4.2s | Remaining: 6.0s
  Processed 600,000/962,990 (62.3%) | Elapsed: 4.6s | Remaining: 2.8s
  Processed 800,000/962,990 (83.1%) | Elapsed: 5.1s | Remaining: 1.0s

 Calculated all metric points in 5.7 seconds

METRIC POINTS SUMMARY
finishing_points              : Total=   14,965 | Positive=  8,069 | Negative=     0
chance_creation_points        : Total=   34,328 | Positive=  4,802 | Negative=     0
ball_progression_points       : Total=  159,290 | Positive= 76,225 | Negative=     0
dribbling_points              : Total=   33,689 | Positive= 21,656 | Negative=     0
ball_winning_points           : Total=   77,489 | Positive= 34,929 | Negative=     0
defensive_actions_points      : Total=   76,371 | Positive= 59,730 | Negative=     0
passing_accuracy_points       : Tot

In [7]:
# Cell 6: Load complete player names from all 306 matches

print("\n" + "="*70)
print("LOADING COMPLETE PLAYER NAMES")
print("="*70)

player_names_cache = PROCESSED_DIR / "player_names_complete.parquet"

if player_names_cache.exists():
    print(f" Found cached player names")
    player_names_complete = pl.read_parquet(player_names_cache)
    print(f"   Loaded {len(player_names_complete)} player names from cache")
else:
    print(f"⏳ Extracting player names from all 306 matches...")
    print("   This will take ~5-7 minutes...")
    
    player_names_dict = {}
    failed_count = 0
    
    for match_id in tqdm(matches['matchId'], desc="Extracting names"):
        try:
            dataset = impect.load_open_data(match_id=match_id, competition_id=743)
            
            for team in dataset.metadata.teams:
                for player in team.players:
                    player_id_str = str(player.player_id)
                    
                    if player_id_str not in player_names_dict:
                        player_names_dict[player_id_str] = {
                            'player_name': player.name if hasattr(player, 'name') else None,
                            'jersey_no': player.jersey_no if hasattr(player, 'jersey_no') else None,
                        }
        except Exception as e:
            failed_count += 1
            continue
    
    player_names_complete = pl.DataFrame([
        {
            'player_id': pid,
            'player_name': info['player_name'],
            'jersey_no': info['jersey_no']
        }
        for pid, info in player_names_dict.items()
    ])
    
    # Save cache
    player_names_complete.write_parquet(player_names_cache)
    
    print(f"\n Extracted {len(player_names_complete)} player names")
    print(f"   Failed matches: {failed_count}")
    print(f"   Saved to: {player_names_cache}")

# Check for nulls
null_names = player_names_complete.filter(pl.col('player_name').is_null())
print(f"\n Player names quality:")
print(f"   Total: {len(player_names_complete)}")
print(f"   With names: {len(player_names_complete) - len(null_names)}")
print(f"   Null names: {len(null_names)}")

if len(null_names) > 0:
    print(f"\n  Some players still have null names (data quality issue)")
    print(null_names.head(5))

print("\n Player names loaded and ready")


LOADING COMPLETE PLAYER NAMES
 Found cached player names
   Loaded 570 player names from cache

 Player names quality:
   Total: 570
   With names: 570
   Null names: 0

 Player names loaded and ready


In [8]:
# Cell 7: Classify player positions based on average field position

print("\n" + "="*70)
print("CLASSIFYING PLAYER POSITIONS")
print("="*70)

# Calculate season-average position for each player
player_positions = (
    all_events
    .filter(pl.col('player_id').is_not_null())
    .group_by('player_id')
    .agg([
        pl.mean('coordinates_x').alias('season_avg_x'),
        pl.mean('coordinates_y').alias('season_avg_y'),
        pl.len().alias('total_events'),
    ])
)

print(f" Calculated average positions for {len(player_positions)} players")

# Classify position based on season average x-coordinate
def classify_position(avg_x):
    """
    Classify player position based on average x-coordinate.
    
    Thresholds (secondspectrum coordinates):
    - Forward: x > 10 (primarily in opponent's half)
    - Midfielder: -10 ≤ x ≤ 10 (central areas)
    - Defender: x < -10 (primarily in own half)
    
    Args:
        avg_x: Average x-coordinate across all events
        
    Returns:
        str: 'forward', 'midfielder', or 'defender'
    """
    if avg_x is None:
        return 'midfielder'  # Default for missing data
    
    if avg_x > 10:
        return 'forward'
    elif avg_x < -10:
        return 'defender'
    else:
        return 'midfielder'

# Apply classification
positions = [classify_position(x) for x in player_positions['season_avg_x']]
player_positions = player_positions.with_columns([
    pl.Series('position', positions)
])

# Distribution
print("\n Position distribution:")
position_dist = player_positions.group_by('position').agg([
    pl.len().alias('count'),
    pl.mean('season_avg_x').alias('avg_x'),
    pl.min('season_avg_x').alias('min_x'),
    pl.max('season_avg_x').alias('max_x'),
]).sort('position')

print(position_dist)

print("\n Position classification complete")


CLASSIFYING PLAYER POSITIONS
 Calculated average positions for 493 players

 Position distribution:
shape: (3, 5)
┌────────────┬───────┬────────────┬────────────┬────────────┐
│ position   ┆ count ┆ avg_x      ┆ min_x      ┆ max_x      │
│ ---        ┆ ---   ┆ ---        ┆ ---        ┆ ---        │
│ str        ┆ u32   ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪═══════╪════════════╪════════════╪════════════╡
│ defender   ┆ 120   ┆ -22.882705 ┆ -43.794366 ┆ -10.444847 │
│ forward    ┆ 121   ┆ 14.079463  ┆ 10.031763  ┆ 41.833333  │
│ midfielder ┆ 252   ┆ 1.127405   ┆ -9.782953  ┆ 9.987798   │
└────────────┴───────┴────────────┴────────────┴────────────┘

 Position classification complete


In [9]:
# Cell 8: Aggregate all metric points by player

print("\n" + "="*70)
print("AGGREGATING METRICS BY PLAYER")
print("="*70)

metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

# Aggregate by player
player_aggregated = (
    all_events
    .filter(pl.col('player_id').is_not_null())
    .group_by('player_id')
    .agg([
        # Sum all metric points
        pl.sum('finishing_points').alias('total_finishing'),
        pl.sum('chance_creation_points').alias('total_chance_creation'),
        pl.sum('ball_progression_points').alias('total_ball_progression'),
        pl.sum('dribbling_points').alias('total_dribbling'),
        pl.sum('ball_winning_points').alias('total_ball_winning'),
        pl.sum('defensive_actions_points').alias('total_defensive_actions'),
        pl.sum('passing_accuracy_points').alias('total_passing_accuracy'),
        pl.sum('long_passing_points').alias('total_long_passing'),
        
        # Count events and matches
        pl.len().alias('total_events'),
        pl.col('match_id').n_unique().alias('matches_played'),
    ])
)

print(f" Aggregated metrics for {len(player_aggregated)} players")

# Join with player names
player_aggregated = player_aggregated.join(
    player_names_complete.select(['player_id', 'player_name']),
    on='player_id',
    how='left'
)

# Join with positions
player_aggregated = player_aggregated.join(
    player_positions.select(['player_id', 'position', 'season_avg_x']),
    on='player_id',
    how='left'
)

# Verify no duplicates
print(f"\n Verification:")
print(f"   Total rows: {len(player_aggregated)}")
print(f"   Unique player_ids: {player_aggregated['player_id'].n_unique()}")

assert len(player_aggregated) == player_aggregated['player_id'].n_unique(), "❌ DUPLICATES FOUND!"

print(f"   No duplicates! ")

# Show sample
print("\n Sample of aggregated data:")
print(player_aggregated.select([
    'player_name', 'position', 'matches_played', 'total_events',
    'total_finishing', 'total_ball_progression', 'total_ball_winning'
]).head(5))


AGGREGATING METRICS BY PLAYER
 Aggregated metrics for 493 players

 Verification:
   Total rows: 493
   Unique player_ids: 493
   No duplicates! 

 Sample of aggregated data:
shape: (5, 7)
┌──────────────┬────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ player_name  ┆ position   ┆ matches_play ┆ total_event ┆ total_finis ┆ total_ball_ ┆ total_ball_ │
│ ---          ┆ ---        ┆ ed           ┆ s           ┆ hing        ┆ progression ┆ winning     │
│ str          ┆ str        ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆            ┆ u32          ┆ u32         ┆ f64         ┆ f64         ┆ f64         │
╞══════════════╪════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Diadié       ┆ defender   ┆ 1            ┆ 7           ┆ 0.0         ┆ 0.0         ┆ 2.5         │
│ Samassékou   ┆            ┆              ┆             ┆             ┆             ┆             │
│ 

In [10]:
# Cell 9: Estimate minutes played and calculate per-90 metrics (FIXED)

print("\n" + "="*70)
print("CALCULATING PER-90 METRICS")
print("="*70)

# Estimate minutes based on event participation
# Calculate average events per match
avg_events_per_match = all_events.group_by('match_id').agg(
    pl.len().alias('events')
)['events'].mean()  # Extract the value properly

print(f"Average events per match: {avg_events_per_match:.0f}")

avg_starter_events_per_match = 35  # Rough baseline for full-90 player

player_aggregated = player_aggregated.with_columns([
    # Events per match for this player
    (pl.col('total_events') / pl.col('matches_played')).alias('events_per_match'),
])

player_aggregated = player_aggregated.with_columns([
    # Estimate minutes per match (capped at 90)
    (
        (pl.col('events_per_match') / avg_starter_events_per_match) * 90
    ).clip(0, 90).alias('estimated_minutes_per_match'),
])

player_aggregated = player_aggregated.with_columns([
    # Total estimated minutes
    (pl.col('estimated_minutes_per_match') * pl.col('matches_played')).alias('estimated_total_minutes')
])

print(f"\n Estimated playing time for all players")

# Minutes distribution
print(f"\n Minutes distribution:")
print(f"   Min: {player_aggregated['estimated_total_minutes'].min():.0f} minutes")
print(f"   Max: {player_aggregated['estimated_total_minutes'].max():.0f} minutes")
print(f"   Mean: {player_aggregated['estimated_total_minutes'].mean():.0f} minutes")
print(f"   Median: {player_aggregated['estimated_total_minutes'].median():.0f} minutes")

# Filter for players with substantial minutes (>500)
MIN_MINUTES = 500

player_aggregated = player_aggregated.filter(
    pl.col('estimated_total_minutes') >= MIN_MINUTES
)

print(f"\n Filtered to {len(player_aggregated)} players with >{MIN_MINUTES} minutes")

# Calculate per-90 for each metric
metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

for metric in metric_columns:
    player_aggregated = player_aggregated.with_columns([
        ((pl.col(f'total_{metric}') / pl.col('estimated_total_minutes')) * 90)
        .alias(f'{metric}_per90')
    ])

print(f"\n Calculated per-90 for all 8 metrics")

# Show per-90 distributions
print(f"\n Per-90 Metric Distributions:")
for metric in metric_columns:
    col = f'{metric}_per90'
    mean_val = player_aggregated[col].mean()
    max_val = player_aggregated[col].max()
    min_val = player_aggregated[col].min()
    print(f"   {metric:25s}: Range=[{min_val:>5.2f}, {max_val:>5.2f}], Mean={mean_val:>5.2f}")

print("\n Top 5 by finishing per-90:")
print(player_aggregated.select([
    'player_name', 'position', 'finishing_per90', 'matches_played', 'estimated_total_minutes'
]).sort('finishing_per90', descending=True).head(5))

print("\n Top 5 by ball progression per-90:")
print(player_aggregated.select([
    'player_name', 'position', 'ball_progression_per90', 'matches_played', 'estimated_total_minutes'
]).sort('ball_progression_per90', descending=True).head(5))


CALCULATING PER-90 METRICS
Average events per match: 3147

 Estimated playing time for all players

 Minutes distribution:
   Min: 8 minutes
   Max: 3060 minutes
   Mean: 1694 minutes
   Median: 1890 minutes

 Filtered to 395 players with >500 minutes

 Calculated per-90 for all 8 metrics

 Per-90 Metric Distributions:
   finishing                : Range=[ 0.00, 13.83], Mean= 1.46
   chance_creation          : Range=[ 0.00, 22.16], Mean= 3.47
   ball_progression         : Range=[ 1.45, 55.58], Mean=16.36
   dribbling                : Range=[ 0.34, 10.07], Mean= 3.42
   ball_winning             : Range=[ 0.00, 23.21], Mean= 8.05
   defensive_actions        : Range=[ 0.48, 24.84], Mean= 8.02
   passing_accuracy         : Range=[ 2.53, 77.04], Mean=16.83
   long_passing             : Range=[-0.48, 14.52], Mean= 3.03

 Top 5 by finishing per-90:
shape: (5, 5)
┌─────────────────┬──────────┬─────────────────┬────────────────┬─────────────────────────┐
│ player_name     ┆ position ┆ finishin

In [11]:
# Cell 10: Calculate within-position percentile ratings (0-99 scale)

print("\n" + "="*70)
print("CALCULATING WITHIN-POSITION PERCENTILE RATINGS (0-99)")
print("="*70)

from scipy.stats import percentileofscore

# Define metric columns
metric_columns = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

# For each metric, calculate percentile WITHIN each position
print("\nCalculating percentiles within positions...")

for metric in metric_columns:
    per90_col = f'{metric}_per90'
    rating_col = f'{metric}_rating'
    
    # Store ratings for all players
    all_ratings = []
    
    # Process each position separately
    for position in ['forward', 'midfielder', 'defender']:
        # Get players in this position
        position_players = player_aggregated.filter(pl.col('position') == position)
        
        if len(position_players) == 0:
            continue
        
        # Get per-90 values for this position
        per90_values = position_players[per90_col].to_numpy()
        
        # Calculate percentile for each player WITHIN their position
        percentiles = []
        for value in per90_values:
            # percentileofscore returns 0-100
            pct = percentileofscore(per90_values, value, kind='rank')
            percentiles.append(pct)
        
        # Scale to 0-99 (cap at 99, no perfect 100s)
        percentiles_scaled = [p * 0.99 for p in percentiles]
        
        # Store with player_ids
        for i, player_id in enumerate(position_players['player_id']):
            all_ratings.append({
                'player_id': player_id,
                rating_col: percentiles_scaled[i]
            })
    
    # Join ratings back to main dataframe
    ratings_df = pl.DataFrame(all_ratings)
    player_aggregated = player_aggregated.join(ratings_df, on='player_id', how='left')
    
    # Show distribution for this metric
    position_stats = player_aggregated.group_by('position').agg([
        pl.col(rating_col).mean().alias('mean'),
        pl.col(rating_col).min().alias('min'),
        pl.col(rating_col).max().alias('max'),
    ]).sort('position')
    
    print(f"\n{metric.upper().replace('_', ' ')} RATING:")
    print(position_stats)

print("\n" + "="*70)
print(" ALL 8 METRICS SCALED TO 0-99 (WITHIN POSITION)")
print("="*70)

# Verify all ratings are 0-99
print("\nVerification - Rating ranges:")
for metric in metric_columns:
    rating_col = f'{metric}_rating'
    min_r = player_aggregated[rating_col].min()
    max_r = player_aggregated[rating_col].max()
    mean_r = player_aggregated[rating_col].mean()
    
    # Check if any exceed 99
    over_99 = (player_aggregated[rating_col] > 99).sum()
    
    print(f"   {metric:25s}: [{min_r:>5.1f}, {max_r:>5.1f}], Mean={mean_r:>5.1f}, Over 99: {over_99}")

print("\n All ratings properly bounded to 0-99 range")

# Show top performers in each metric
print("\n" + "="*70)
print("TOP 5 BY EACH RATING (Within-Position Percentiles)")
print("="*70)

for metric in metric_columns:
    rating_col = f'{metric}_rating'
    per90_col = f'{metric}_per90'
    
    print(f"\n{metric.upper().replace('_', ' ')}:")
    print(player_aggregated.select([
        'player_name', 'position', rating_col, per90_col, 'matches_played'
    ]).sort(rating_col, descending=True).head(5))


CALCULATING WITHIN-POSITION PERCENTILE RATINGS (0-99)

Calculating percentiles within positions...

FINISHING RATING:
shape: (3, 4)
┌────────────┬───────────┬───────────┬──────┐
│ position   ┆ mean      ┆ min       ┆ max  │
│ ---        ┆ ---       ┆ ---       ┆ ---  │
│ str        ┆ f64       ┆ f64       ┆ f64  │
╞════════════╪═══════════╪═══════════╪══════╡
│ defender   ┆ 49.990099 ┆ 15.683168 ┆ 99.0 │
│ forward    ┆ 50.068966 ┆ 1.137931  ┆ 99.0 │
│ midfielder ┆ 49.73913  ┆ 0.717391  ┆ 99.0 │
└────────────┴───────────┴───────────┴──────┘

CHANCE CREATION RATING:
shape: (3, 4)
┌────────────┬───────────┬──────────┬──────┐
│ position   ┆ mean      ┆ min      ┆ max  │
│ ---        ┆ ---       ┆ ---      ┆ ---  │
│ str        ┆ f64       ┆ f64      ┆ f64  │
╞════════════╪═══════════╪══════════╪══════╡
│ defender   ┆ 49.990099 ┆ 14.70297 ┆ 99.0 │
│ forward    ┆ 50.068966 ┆ 2.275862 ┆ 99.0 │
│ midfielder ┆ 49.73913  ┆ 0.717391 ┆ 99.0 │
└────────────┴───────────┴──────────┴──────┘

BALL PRO

In [12]:
# Cell 10: Calculate within-position percentile ratings 

print("\n" + "="*70)
print("CALCULATING WITHIN-POSITION PERCENTILE RATINGS (0-99)")
print("="*70)

from scipy.stats import percentileofscore

# =============================================================================
# DEFINE POSITION-RELEVANT METRICS
# =============================================================================

POSITION_RELEVANT_METRICS = {
    'forward': ['finishing', 'chance_creation', 'dribbling', 'ball_progression'],
    'midfielder': ['finishing', 'chance_creation', 'ball_progression', 'dribbling',
                   'ball_winning', 'defensive_actions', 'passing_accuracy'],
    'defender': ['ball_winning', 'defensive_actions', 'ball_progression',
                 'passing_accuracy', 'long_passing']
}

print(" Position-Relevant Metrics:")
for pos, metrics in POSITION_RELEVANT_METRICS.items():
    print(f"  {pos:10s} ({len(metrics)} metrics): {', '.join(metrics)}")

all_metrics = [
    'finishing', 'chance_creation', 'ball_progression', 'dribbling',
    'ball_winning', 'defensive_actions', 'passing_accuracy', 'long_passing'
]

# =============================================================================
# CALCULATE RATINGS - ONE METRIC AT A TIME
# =============================================================================

print("\n Calculating percentiles within positions...")

for metric in all_metrics:
    per90_col = f'{metric}_per90'
    rating_col = f'{metric}_rating'
    
    # Initialize rating column with NULL for all players
    player_aggregated = player_aggregated.with_columns([
        pl.lit(None, dtype=pl.Float64).alias(rating_col)
    ])
    
    # Process each position separately
    for position in ['forward', 'midfielder', 'defender']:
        # Skip if metric not relevant for this position
        if metric not in POSITION_RELEVANT_METRICS[position]:
            continue
        
        # Get players in this position
        position_mask = player_aggregated['position'] == position
        position_indices = [i for i, val in enumerate(position_mask) if val]
        
        if len(position_indices) == 0:
            continue
        
        # Get per-90 values for this position
        position_players = player_aggregated.filter(pl.col('position') == position)
        per90_values = position_players[per90_col].to_numpy()
        
        # Calculate percentiles
        percentiles = []
        for value in per90_values:
            pct = percentileofscore(per90_values, value, kind='rank')
            scaled = round(pct * 0.99, 1)  # Scale to 0-99, round to 1 decimal
            percentiles.append(scaled)
        
        # Update ratings for this position using row indices
        current_ratings = player_aggregated[rating_col].to_list()
        
        for i, pos_idx in enumerate(position_indices):
            current_ratings[pos_idx] = percentiles[i]
        
        # Replace column
        player_aggregated = player_aggregated.with_columns([
            pl.Series(rating_col, current_ratings)
        ])
    
    print(f"   {metric}")

print("\n All percentile ratings calculated")

# =============================================================================
# VERIFICATION
# =============================================================================

print("\n" + "="*70)
print("VERIFICATION: Ratings by Position")
print("="*70)

for position in ['forward', 'midfielder', 'defender']:
    pos_players = player_aggregated.filter(pl.col('position') == position)
    
    print(f"\n{position.upper()}S ({len(pos_players)} players):")
    
    for metric in all_metrics:
        rating_col = f'{metric}_rating'
        non_null = pos_players[rating_col].drop_nulls()
        
        if len(non_null) > 0:
            mean_r = non_null.mean()
            max_r = non_null.max()
            min_r = non_null.min()
            print(f"   {metric:25s}: Mean={mean_r:>5.1f}, Range=[{min_r:>5.1f}, {max_r:>5.1f}]")
        else:
            print(f"    {metric:25s}: NULL (not relevant)")

# Verify bounds
print("\n" + "="*70)
print("VERIFICATION: All Ratings 0-99")
print("="*70)

all_good = True
for metric in all_metrics:
    rating_col = f'{metric}_rating'
    non_null = player_aggregated[rating_col].drop_nulls()
    
    if len(non_null) > 0:
        max_r = non_null.max()
        over_99 = (non_null > 99.0).sum()
        
        if max_r <= 99.0 and over_99 == 0:
            print(f" {metric:25s}: Max={max_r:>5.1f}, Over 99: {over_99}")
        else:
            print(f" {metric:25s}: Max={max_r:>5.1f}, Over 99: {over_99}")
            all_good = False

if all_good:
    print("\n ALL RATINGS PROPERLY BOUNDED TO 0-99!")
else:
    print("\n Some ratings exceed 99 - check logic!")

# Show top 5 for each metric
print("\n" + "="*70)
print("TOP 5 BY EACH RATING")
print("="*70)

for metric in all_metrics:
    rating_col = f'{metric}_rating'
    per90_col = f'{metric}_per90'
    
    print(f"\n{metric.upper().replace('_', ' ')}:")
    print(player_aggregated
          .filter(pl.col(rating_col).is_not_null())
          .select(['player_name', 'position', rating_col, per90_col, 'matches_played'])
          .sort(rating_col, descending=True)
          .head(5))

print("\n Cell 10 Complete!")


CALCULATING WITHIN-POSITION PERCENTILE RATINGS (0-99)
 Position-Relevant Metrics:
  forward    (4 metrics): finishing, chance_creation, dribbling, ball_progression
  midfielder (7 metrics): finishing, chance_creation, ball_progression, dribbling, ball_winning, defensive_actions, passing_accuracy
  defender   (5 metrics): ball_winning, defensive_actions, ball_progression, passing_accuracy, long_passing

 Calculating percentiles within positions...
   finishing
   chance_creation
   ball_progression
   dribbling
   ball_winning
   defensive_actions
   passing_accuracy
   long_passing

 All percentile ratings calculated

VERIFICATION: Ratings by Position

FORWARDS (87 players):
   finishing                : Mean= 50.1, Range=[  1.1,  99.0]
   chance_creation          : Mean= 50.1, Range=[  2.3,  99.0]
   ball_progression         : Mean= 50.1, Range=[  1.1,  99.0]
   dribbling                : Mean= 50.1, Range=[  1.1,  99.0]
    ball_winning             : NULL (not relevant)
    defensiv

In [13]:
# Cell 11: Calculate overall ratings (cross-position + within-position)

print("\n" + "="*70)
print("CALCULATING OVERALL RATINGS")
print("="*70)

# =============================================================================
# DEFINE POSITION-SPECIFIC WEIGHTS FOR OVERALL RATING
# =============================================================================

# Weights sum to 1.0, using ONLY relevant metrics per position
POSITION_OVERALL_WEIGHTS = {
    'forward': {
        'finishing': 0.45,
        'chance_creation': 0.30,
        'dribbling': 0.20,
        'ball_progression': 0.05,
    },
    'midfielder': {
        'ball_progression': 0.30,
        'chance_creation': 0.20,
        'passing_accuracy': 0.15,
        'ball_winning': 0.15,
        'dribbling': 0.10,
        'defensive_actions': 0.05,
        'finishing': 0.05,
    },
    'defender': {
        'ball_winning': 0.35,
        'defensive_actions': 0.30,
        'ball_progression': 0.20,
        'passing_accuracy': 0.10,
        'long_passing': 0.05,
    }
}

print("Position-specific weights for overall rating:")
for position, weights in POSITION_OVERALL_WEIGHTS.items():
    print(f"\n{position.upper()}:")
    total_weight = sum(weights.values())
    for metric, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        print(f"  {metric:25s}: {weight:>5.1%}")
    print(f"  {'TOTAL':25s}: {total_weight:>5.1%}")

# =============================================================================
# CALCULATE CROSS-POSITION OVERALL RATING
# =============================================================================

print("\n Calculating cross-position overall ratings...")

def calculate_cross_position_overall(row):
    """
    Calculate overall rating using position-specific weights.
    Allows comparison across all positions (for market value analysis).
    """
    position = row['position']
    
    if position not in POSITION_OVERALL_WEIGHTS:
        return None
    
    weights = POSITION_OVERALL_WEIGHTS[position]
    overall = 0.0
    
    for metric, weight in weights.items():
        rating = row.get(f'{metric}_rating')
        if rating is not None:
            overall += rating * weight
    
    return round(overall, 1)  # Round to 1 decimal

# Calculate for all players
cross_position_overall = []
for row in player_aggregated.iter_rows(named=True):
    cross_position_overall.append(calculate_cross_position_overall(row))

player_aggregated = player_aggregated.with_columns([
    pl.Series('overall_rating', cross_position_overall)
])

print(" Cross-position overall ratings calculated")

# Distribution
print(f"\n Cross-Position Overall Rating Distribution:")
print(f"   Min: {player_aggregated['overall_rating'].min():.1f}")
print(f"   Max: {player_aggregated['overall_rating'].max():.1f}")
print(f"   Mean: {player_aggregated['overall_rating'].mean():.1f}")
print(f"   Median: {player_aggregated['overall_rating'].median():.1f}")

# =============================================================================
# CALCULATE WITHIN-POSITION OVERALL RATING
# =============================================================================

print("\n Calculating within-position overall percentiles...")

# For each position, calculate percentile of overall rating
within_position_overall = []

for position in ['forward', 'midfielder', 'defender']:
    position_players = player_aggregated.filter(pl.col('position') == position)
    
    if len(position_players) == 0:
        continue
    
    # Get overall ratings for this position
    overall_values = position_players['overall_rating'].to_numpy()
    
    # Calculate percentile within position
    for i, row in enumerate(position_players.iter_rows(named=True)):
        overall_val = overall_values[i]
        pct = percentileofscore(overall_values, overall_val, kind='rank')
        scaled = round(pct * 0.99, 1)  # 0-99 scale
        
        within_position_overall.append({
            'player_id': row['player_id'],
            'within_position_overall': scaled
        })

# Join back
within_df = pl.DataFrame(within_position_overall)
player_aggregated = player_aggregated.join(within_df, on='player_id', how='left')

print(" Within-position overall percentiles calculated")

# =============================================================================
# SHOW RESULTS
# =============================================================================

print("\n" + "="*70)
print("TOP 20 PLAYERS - CROSS-POSITION OVERALL RATING")
print("="*70)
print("(Used for market value comparison - all positions on same scale)")
print()

top_20_cross = player_aggregated.select([
    'player_name', 'position', 'overall_rating', 'within_position_overall',
    'finishing_rating', 'chance_creation_rating', 'ball_progression_rating',
    'ball_winning_rating', 'matches_played'
]).sort('overall_rating', descending=True).head(20)

print(top_20_cross)

# Position distribution in top 20
print("\n Position distribution in top 20:")
pos_dist = top_20_cross.group_by('position').agg(pl.len().alias('count')).sort('position')
print(pos_dist)

# Top 10 per position by within-position overall
print("\n" + "="*70)
print("TOP 10 BY POSITION - WITHIN-POSITION OVERALL")
print("="*70)
print("(Best in their role - 99 = best forward, best midfielder, best defender)")
print()

for position in ['forward', 'midfielder', 'defender']:
    print(f"\n{position.upper()}S:")
    print(player_aggregated
          .filter(pl.col('position') == position)
          .select(['player_name', 'within_position_overall', 'overall_rating', 'matches_played'])
          .sort('within_position_overall', descending=True)
          .head(10))

print("\n Cell 11 Complete - Both overall ratings calculated!")


CALCULATING OVERALL RATINGS
Position-specific weights for overall rating:

FORWARD:
  finishing                : 45.0%
  chance_creation          : 30.0%
  dribbling                : 20.0%
  ball_progression         :  5.0%
  TOTAL                    : 100.0%

MIDFIELDER:
  ball_progression         : 30.0%
  chance_creation          : 20.0%
  passing_accuracy         : 15.0%
  ball_winning             : 15.0%
  dribbling                : 10.0%
  defensive_actions        :  5.0%
  finishing                :  5.0%
  TOTAL                    : 100.0%

DEFENDER:
  ball_winning             : 35.0%
  defensive_actions        : 30.0%
  ball_progression         : 20.0%
  passing_accuracy         : 10.0%
  long_passing             :  5.0%
  TOTAL                    : 100.0%

 Calculating cross-position overall ratings...
 Cross-position overall ratings calculated

 Cross-Position Overall Rating Distribution:
   Min: 2.0
   Max: 92.6
   Mean: 49.9
   Median: 48.9

 Calculating within-position o

In [14]:
top_100 = player_aggregated.sort('overall_rating', descending=True).head(100)
top_100["player_name"].head(100)

player_name
str
"""Andrej Kramaric"""
"""Timo Hübers"""
"""Min-jae Kim"""
"""Florian Wirtz"""
"""Nico Schlotterbeck"""
…
"""Eric Dier"""
"""Kiliann Sildillia"""
"""Jadon Sancho"""


In [15]:
# Cell 12A: Generate top 100 player list for market value lookup

print("Generating player list for market value lookup...")

# Get top 100 by overall rating
top_300_lookup = player_aggregated.sort('overall_rating', descending=True).head(300)

# Save as plain text file (one name per line)
lookup_file = DATA_DIR / "top_300_for_market_lookup.txt"

with open(lookup_file, 'w', encoding='utf-8') as f:
    for name in top_300_lookup['player_name']:
        f.write(f"{name}\n")

print(f" Saved {len(top_300_lookup)} player names to:")
print(f"   {lookup_file.absolute()}")
print(f"\nFile location: {lookup_file.absolute()}")

# Also save with additional info for reference
lookup_with_info = DATA_DIR / "top_300_with_ratings.csv"
top_300_lookup.select([
    'player_name', 'position', 'overall_rating', 'matches_played'
]).write_csv(lookup_with_info)

print(f"\n Also saved detailed version to:")
print(f"   {lookup_with_info.absolute()}")

Generating player list for market value lookup...
 Saved 300 player names to:
   /Users/tanishbhilare/Desktop/SoccerImpectHackathon/Notebooks/../data/top_300_for_market_lookup.txt

File location: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/Notebooks/../data/top_300_for_market_lookup.txt

 Also saved detailed version to:
   /Users/tanishbhilare/Desktop/SoccerImpectHackathon/Notebooks/../data/top_300_with_ratings.csv


In [16]:
# Cell 12: Load market values and calculate value scores 

print("\n" + "="*70)
print("LOADING MARKET VALUES & CALCULATING VALUE SCORES")
print("="*70)

# Load market values
market_values_file = DATA_DIR / "market_values.csv"

if market_values_file.exists():
    print(f" Loading market values from: {market_values_file}")
    
    # Load (auto-detects numeric)
    market_values = pl.read_csv(market_values_file)
    print(f"   Total entries: {len(market_values)}")
    
    # Ensure numeric type
    market_values = market_values.with_columns([
        pl.col('market_value_millions').cast(pl.Float64, strict=False)
    ])
    
    market_values_clean = market_values.filter(
        pl.col('market_value_millions').is_not_null()
    )
    
    print(f"   Clean numeric values: {len(market_values_clean)}")
    
    # Sample
    print(f"\n Sample of market values:")
    print(market_values_clean.head(10))
    
    # Join with ratings
    player_final = player_aggregated.join(
        market_values_clean,
        on='player_name',
        how='left'
    )
    
    matched = player_final.filter(pl.col('market_value_millions').is_not_null())
    print(f"\n Matched {len(matched)}/{len(player_aggregated)} players ({len(matched)/len(player_aggregated)*100:.1f}% coverage)")
    
    # Calculate value metrics
    player_final = player_final.with_columns([
        # Value Score
        (pl.col('overall_rating') / pl.col('market_value_millions')).alias('value_score'),
        
        # Expected rating (€15M = 50 rating baseline)
        # Fixed: clip(min, max) not clip(upper=)
        (
            (pl.col('market_value_millions') / 15.0) * 50
        ).clip(0, 95).alias('expected_rating'),
        
        # Rating vs Expected
        (
            pl.col('overall_rating') - 
            ((pl.col('market_value_millions') / 15.0) * 50).clip(0, 95)
        ).alias('rating_vs_expected'),
    ])
    
    print("\n✅ Value metrics calculated")
    
    # Stats
    print(f"\n Market Value Distribution:")
    print(f"   Range: €{matched['market_value_millions'].min():.1f}M - €{matched['market_value_millions'].max():.1f}M")
    print(f"   Mean: €{matched['market_value_millions'].mean():.1f}M")
    print(f"   Median: €{matched['market_value_millions'].median():.1f}M")
    
    print(f"\n Value Score Distribution:")
    valid_scores = player_final.filter(pl.col('value_score').is_not_null())['value_score']
    print(f"   Range: {valid_scores.min():.2f} - {valid_scores.max():.2f} rating/€M")
    print(f"   Mean: {valid_scores.mean():.2f} rating/€M")
    
    # Top 10 best value
    print("\n" + "="*70)
    print(" TOP 10 BEST VALUE PLAYERS (Hidden Gems!)")
    print("="*70)
    
    best_value = player_final.filter(
        pl.col('value_score').is_not_null()
    ).select([
        'player_name', 'position', 'overall_rating', 
        'market_value_millions', 'value_score', 
        'rating_vs_expected', 'matches_played'
    ]).sort('value_score', descending=True).head(10)  # descending=True for highest first
    
    print(best_value)
    
    # Top 10 worst value  
    print("\n" + "="*70)
    print("  TOP 10 WORST VALUE PLAYERS (Overpriced)")
    print("="*70)
    
    worst_value = player_final.filter(
        (pl.col('value_score').is_not_null()) &
        (pl.col('market_value_millions') >= 15)
    ).select([
        'player_name', 'position', 'overall_rating', 
        'market_value_millions', 'value_score',
        'rating_vs_expected', 'matches_played'
    ]).sort('value_score', descending=False).head(10)  # descending=False for lowest first
    
    print(worst_value)
    
else:
    print(f" File not found: {market_values_file}")
    player_final = player_aggregated

print("\n Cell 12 complete")


LOADING MARKET VALUES & CALCULATING VALUE SCORES
 Loading market values from: ../data/market_values.csv
   Total entries: 300
   Clean numeric values: 300

 Sample of market values:
shape: (10, 2)
┌────────────────────┬───────────────────────┐
│ player_name        ┆ market_value_millions │
│ ---                ┆ ---                   │
│ str                ┆ f64                   │
╞════════════════════╪═══════════════════════╡
│ Andrej Kramaric    ┆ 6.0                   │
│ Timo Hübers        ┆ 5.5                   │
│ Min-jae Kim        ┆ 45.0                  │
│ Florian Wirtz      ┆ 130.0                 │
│ Nico Schlotterbeck ┆ 40.0                  │
│ Patrick Mainka     ┆ 3.5                   │
│ Willian Pacho      ┆ 35.0                  │
│ Kevin Stöger       ┆ 5.0                   │
│ Leroy Sané         ┆ 70.0                  │
│ Dan-Axel Zagadou   ┆ 12.0                  │
└────────────────────┴───────────────────────┘

 Matched 300/395 players (75.9% coverage)

✅ Valu

In [17]:
# Cell 13: Identify true hidden gems and create categories

print("\n" + "="*70)
print("CATEGORIZING PLAYERS BY VALUE PROPOSITION")
print("="*70)

# Create value categories
player_final = player_final.with_columns([
    pl.when(
        (pl.col('overall_rating') >= 85) & (pl.col('market_value_millions') < 15)
    ).then(pl.lit(" Hidden Gem (Elite Performance, Low Price)"))
    
    .when(
        (pl.col('overall_rating') >= 75) & (pl.col('market_value_millions') < 20)
    ).then(pl.lit(" Great Value (High Performance, Reasonable Price)"))
    
    .when(
        (pl.col('overall_rating') >= 85) & (pl.col('market_value_millions') >= 50)
    ).then(pl.lit(" Elite (Expensive but Worth It)"))
    
    .when(
        (pl.col('overall_rating') < 70) & (pl.col('market_value_millions') >= 30)
    ).then(pl.lit("  Overpriced (Expensive, Low Performance)"))
    
    .when(
        (pl.col('overall_rating') >= 70) & (pl.col('market_value_millions') >= 25)
    ).then(pl.lit(" Fair Value (Good Performance, Fair Price)"))
    
    .otherwise(pl.lit(" Standard"))
    
    .alias('value_category')
])

# Show category distribution
print("\n Value Category Distribution:")
categories = player_final.filter(
    pl.col('market_value_millions').is_not_null()
).group_by('value_category').agg([
    pl.len().alias('count'),
    pl.mean('overall_rating').alias('avg_rating'),
    pl.mean('market_value_millions').alias('avg_value')
]).sort('count', descending=True)

print(categories)

# =============================================================================
# HIDDEN GEMS: High Performance (≥75 rating), Low Price (<€15M)
# =============================================================================

print("\n" + "="*70)
print(" HIDDEN GEMS: Elite Performers at Budget Prices")
print("="*70)
print("(Rating ≥75, Market Value <€15M)")
print()

hidden_gems = player_final.filter(
    (pl.col('overall_rating') >= 75) &
    (pl.col('market_value_millions') < 15) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played',
    'finishing_rating', 'chance_creation_rating', 
    'ball_progression_rating', 'ball_winning_rating'
]).sort('overall_rating', descending=True)

print(hidden_gems)
print(f"\n Found {len(hidden_gems)} hidden gems!")

# =============================================================================
# GREAT VALUE: Good Performance (≥70), Affordable (<€20M)
# =============================================================================

print("\n" + "="*70)
print(" GREAT VALUE: Good Performers at Affordable Prices")
print("="*70)
print("(Rating 70-85, Market Value <€20M)")
print()

great_value = player_final.filter(
    (pl.col('overall_rating') >= 70) &
    (pl.col('overall_rating') < 85) &
    (pl.col('market_value_millions') < 20) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score', 'matches_played'
]).sort('overall_rating', descending=True).head(15)

print(great_value)

# =============================================================================
# OVERPRICED: Expensive (≥€30M) but Underperforming (<75 rating)
# =============================================================================

print("\n" + "="*70)
print("  OVERPRICED: Expensive Players Underperforming")
print("="*70)
print("(Rating <75, Market Value ≥€30M)")
print()

overpriced = player_final.filter(
    (pl.col('overall_rating') < 75) &
    (pl.col('market_value_millions') >= 30) &
    (pl.col('market_value_millions').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played'
]).sort('market_value_millions', descending=True)

if len(overpriced) > 0:
    print(overpriced)
else:
    print("   (No players in this category - all expensive players performing well!)")

# =============================================================================
# ELITE VALUE: Top 10 Overall by Value Score (Must be >70 rating)
# =============================================================================

print("\n" + "="*70)
print(" TOP 10 BEST VALUE OVERALL (Rating >70)")
print("="*70)
print("Best performance-to-price ratio for quality players")
print()

elite_value = player_final.filter(
    (pl.col('overall_rating') >= 70) &
    (pl.col('value_score').is_not_null())
).select([
    'player_name', 'position', 'overall_rating', 
    'market_value_millions', 'value_score',
    'rating_vs_expected', 'matches_played'
]).sort('value_score', descending=True).head(10)

print(elite_value)

print("\n Cell 13 complete - Value categories created!")


CATEGORIZING PLAYERS BY VALUE PROPOSITION

 Value Category Distribution:
shape: (6, 4)
┌─────────────────────────────────┬───────┬────────────┬───────────┐
│ value_category                  ┆ count ┆ avg_rating ┆ avg_value │
│ ---                             ┆ ---   ┆ ---        ┆ ---       │
│ str                             ┆ u32   ┆ f64        ┆ f64       │
╞═════════════════════════════════╪═══════╪════════════╪═══════════╡
│  Standard                       ┆ 226   ┆ 52.930973  ┆ 6.89115   │
│  Fair Value (Good Performance,… ┆ 33    ┆ 79.866667  ┆ 41.212121 │
│  Great Value (High Performance… ┆ 21    ┆ 79.295238  ┆ 9.880952  │
│   Overpriced (Expensive, Low P… ┆ 10    ┆ 60.21      ┆ 40.5      │
│  Hidden Gem (Elite Performance… ┆ 6     ┆ 89.85      ┆ 7.0       │
│  Elite (Expensive but Worth It… ┆ 4     ┆ 88.85      ┆ 82.5      │
└─────────────────────────────────┴───────┴────────────┴───────────┘

 HIDDEN GEMS: Elite Performers at Budget Prices
(Rating ≥75, Market Value <€15M)

s

In [18]:
# Cell 14: Save final dataset with all ratings and market values

print("\n" + "="*70)
print("SAVING FINAL DATASET")
print("="*70)

# Save complete dataset
player_final.write_parquet(PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet")
player_final.write_csv(PROCESSED_DIR / "player_ratings_FINAL_with_market_values.csv")

print(f" Saved complete dataset:")
print(f"   {PROCESSED_DIR / 'player_ratings_FINAL_with_market_values.parquet'}")
print(f"   {PROCESSED_DIR / 'player_ratings_FINAL_with_market_values.csv'}")

# Save specific lists for presentation
hidden_gems_only = player_final.filter(
    pl.col('value_category') == "💎 Hidden Gem (Elite Performance, Low Price)"
).select([
    'player_name', 'position', 'overall_rating', 'within_position_overall',
    'market_value_millions', 'value_score', 'rating_vs_expected',
    'finishing_rating', 'chance_creation_rating', 'ball_progression_rating',
    'ball_winning_rating', 'defensive_actions_rating', 'matches_played'
]).sort('overall_rating', descending=True)

hidden_gems_only.write_csv(PROCESSED_DIR / "HIDDEN_GEMS.csv")

print(f"\n Saved hidden gems list:")
print(f"   {PROCESSED_DIR / 'HIDDEN_GEMS.csv'}")
print(f"   {len(hidden_gems_only)} players")

# Save great value list
great_value_only = player_final.filter(
    pl.col('value_category') == "⭐ Great Value (High Performance, Reasonable Price)"
).sort('value_score', descending=True)

great_value_only.write_csv(PROCESSED_DIR / "GREAT_VALUE.csv")

print(f"\n Saved great value list:")
print(f"   {PROCESSED_DIR / 'GREAT_VALUE.csv'}")
print(f"   {len(great_value_only)} players")

# Save overpriced list
overpriced_only = player_final.filter(
    pl.col('value_category') == "⚠️  Overpriced (Expensive, Low Performance)"
).sort('market_value_millions', descending=True)

overpriced_only.write_csv(PROCESSED_DIR / "OVERPRICED.csv")

print(f"\n Saved overpriced list:")
print(f"   {PROCESSED_DIR / 'OVERPRICED.csv'}")
print(f"   {len(overpriced_only)} players")

# Summary
print("\n" + "="*70)
print(" FINAL DATASET SUMMARY")
print("="*70)
print(f"Total players: {len(player_final)}")
print(f"With market values: {len(player_final.filter(pl.col('market_value_millions').is_not_null()))}")
print(f"")
print(f"Value Categories:")
print(f"   Hidden Gems (≥85 rating, <€15M): {len(hidden_gems_only)}")
print(f"   Great Value (70-85 rating, <€20M): {len(great_value_only)}")
print(f"    Overpriced (<75 rating, ≥€30M): {len(overpriced_only)}")
print(f"")
print(f"Rating Metrics: 8 granular + 2 overall")
print(f"Market Coverage: 76% of players")

print("\n" + "="*70)
print(" MILESTONE 1 COMPLETE!")
print("="*70)
print("\nWhat we've built:")
print("   8 granular, interpretable metrics")
print("   Within-position percentile ratings (0-99)")
print("   Two overall ratings (cross-position + within-position)")
print("   Market value integration (300 players)")
print("   Hidden gems identification (6 elite, 21 great value)")
print("   Complete player names (570 players, 0 nulls)")
print("   All data saved and ready for visualization")
print("\nNext Steps:")
print("  → Milestone 2: Create visualizations (radar charts, scatter plots)")
print("  → Milestone 3: Build slide deck")
print("  → Milestone 4: Complete GitHub documentation")


SAVING FINAL DATASET
 Saved complete dataset:
   ../data/processed/player_ratings_FINAL_with_market_values.parquet
   ../data/processed/player_ratings_FINAL_with_market_values.csv

 Saved hidden gems list:
   ../data/processed/HIDDEN_GEMS.csv
   0 players

 Saved great value list:
   ../data/processed/GREAT_VALUE.csv
   0 players

 Saved overpriced list:
   ../data/processed/OVERPRICED.csv
   0 players

 FINAL DATASET SUMMARY
Total players: 395
With market values: 300

Value Categories:
   Hidden Gems (≥85 rating, <€15M): 0
   Great Value (70-85 rating, <€20M): 0
    Overpriced (<75 rating, ≥€30M): 0

Rating Metrics: 8 granular + 2 overall
Market Coverage: 76% of players

 MILESTONE 1 COMPLETE!

What we've built:
   8 granular, interpretable metrics
   Within-position percentile ratings (0-99)
   Two overall ratings (cross-position + within-position)
   Market value integration (300 players)
   Hidden gems identification (6 elite, 21 great value)
   Complete player names (570 players,

In [19]:
# Detailed Musiala analysis
musiala = player_final.filter(pl.col('player_name') == 'Jamal Musiala')

print("JAMAL MUSIALA DETAILED BREAKDOWN:")
print(musiala.select([
    'player_name', 'position', 'overall_rating', 'within_position_overall',
    'finishing_rating', 'chance_creation_rating', 'dribbling_rating',
    'ball_progression_rating', 'matches_played'
]))

JAMAL MUSIALA DETAILED BREAKDOWN:
shape: (1, 9)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ player_na ┆ position ┆ overall_r ┆ within_po ┆ … ┆ chance_cr ┆ dribbling ┆ ball_prog ┆ matches_p │
│ me        ┆ ---      ┆ ating     ┆ sition_ov ┆   ┆ eation_ra ┆ _rating   ┆ ression_r ┆ layed     │
│ ---       ┆ str      ┆ ---       ┆ erall     ┆   ┆ ting      ┆ ---       ┆ ating     ┆ ---       │
│ str       ┆          ┆ f64       ┆ ---       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ u32       │
│           ┆          ┆           ┆ f64       ┆   ┆ f64       ┆           ┆ f64       ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Jamal     ┆ forward  ┆ 81.6      ┆ 87.6      ┆ … ┆ 63.7      ┆ 94.4      ┆ 93.3      ┆ 24        │
│ Musiala   ┆          ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
└───────────┴──────────┴───────────┴───────

In [20]:

# Detailed Musiala analysis
undav = player_final.filter(pl.col('player_name') == 'Deniz Undav')

print("Deniz Undav DETAILED BREAKDOWN:")
print(undav.select([
    'player_name', 'position', 'overall_rating', 'within_position_overall',
    'finishing_rating', 'chance_creation_rating', 'dribbling_rating',
    'ball_progression_rating', 'matches_played'
]))

Deniz Undav DETAILED BREAKDOWN:
shape: (1, 9)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ player_na ┆ position ┆ overall_r ┆ within_po ┆ … ┆ chance_cr ┆ dribbling ┆ ball_prog ┆ matches_p │
│ me        ┆ ---      ┆ ating     ┆ sition_ov ┆   ┆ eation_ra ┆ _rating   ┆ ression_r ┆ layed     │
│ ---       ┆ str      ┆ ---       ┆ erall     ┆   ┆ ting      ┆ ---       ┆ ating     ┆ ---       │
│ str       ┆          ┆ f64       ┆ ---       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ u32       │
│           ┆          ┆           ┆ f64       ┆   ┆ f64       ┆           ┆ f64       ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Deniz     ┆ forward  ┆ 82.7      ┆ 89.9      ┆ … ┆ 68.3      ┆ 76.2      ┆ 79.7      ┆ 30        │
│ Undav     ┆          ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
└───────────┴──────────┴───────────┴─────────

In [21]:
import polars as pl
from pathlib import Path

# Go one level up from notebooks/ to reach the project root
df = pl.read_parquet("../data/processed/player_ratings_FINAL_with_market_values.parquet")
print("ALL COLUMNS:")
print(df.columns)
print("\nSHAPE:", df.shape)

ALL COLUMNS:
['player_id', 'total_finishing', 'total_chance_creation', 'total_ball_progression', 'total_dribbling', 'total_ball_winning', 'total_defensive_actions', 'total_passing_accuracy', 'total_long_passing', 'total_events', 'matches_played', 'player_name', 'position', 'season_avg_x', 'events_per_match', 'estimated_minutes_per_match', 'estimated_total_minutes', 'finishing_per90', 'chance_creation_per90', 'ball_progression_per90', 'dribbling_per90', 'ball_winning_per90', 'defensive_actions_per90', 'passing_accuracy_per90', 'long_passing_per90', 'finishing_rating', 'chance_creation_rating', 'ball_progression_rating', 'dribbling_rating', 'ball_winning_rating', 'defensive_actions_rating', 'passing_accuracy_rating', 'long_passing_rating', 'overall_rating', 'within_position_overall', 'market_value_millions', 'value_score', 'expected_rating', 'rating_vs_expected', 'value_category']

SHAPE: (395, 40)


In [22]:
import os
print("Current working directory:", os.getcwd())

# Search for the parquet file anywhere in your project
for root, dirs, files in os.walk(os.getcwd()):
    for f in files:
        if f.endswith(".parquet"):
            print(os.path.join(root, f))

Current working directory: /Users/tanishbhilare/Desktop/SoccerImpectHackathon/Notebooks


In [23]:
# Check stat columns for a known player
sample = df.filter(pl.col("player_name").str.contains("Kane|Guirassy|Wirtz"))
stat_cols = [c for c in df.columns if any(
    kw in c.lower() for kw in ["goal","assist","shot","pass","carry","duel","def","team","squad","club"]
)]
print("STAT + TEAM COLUMNS FOUND:")
print(stat_cols)
print("\nSAMPLE VALUES:")
print(sample.select(["player_name"] + stat_cols))

STAT + TEAM COLUMNS FOUND:
['total_defensive_actions', 'total_passing_accuracy', 'total_long_passing', 'defensive_actions_per90', 'passing_accuracy_per90', 'long_passing_per90', 'defensive_actions_rating', 'passing_accuracy_rating', 'long_passing_rating']

SAMPLE VALUES:
shape: (3, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ player_na ┆ total_def ┆ total_pas ┆ total_lon ┆ … ┆ long_pass ┆ defensive ┆ passing_a ┆ long_pas │
│ me        ┆ ensive_ac ┆ sing_accu ┆ g_passing ┆   ┆ ing_per90 ┆ _actions_ ┆ ccuracy_r ┆ sing_rat │
│ ---       ┆ tions     ┆ racy      ┆ ---       ┆   ┆ ---       ┆ rating    ┆ ating     ┆ ing      │
│ str       ┆ ---       ┆ ---       ┆ f64       ┆   ┆ f64       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆           ┆   ┆           ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Serh

In [24]:
import os
for root, dirs, files in os.walk("../data"):
    for f in files:
        if f.endswith(".parquet"):
            print(os.path.join(root, f))

../data/processed/squads_metadata.parquet
../data/processed/all_events_with_points.parquet
../data/processed/player_names_complete.parquet
../data/processed/player_ratings_with_8_metrics.parquet
../data/processed/matches_metadata.parquet
../data/processed/all_matches_with_zones.parquet
../data/processed/player_ratings_FINAL_with_market_values.parquet
../data/processed/player_metadata.parquet


In [25]:
squads = pl.read_parquet("../data/processed/squads_metadata.parquet")
print("SQUADS COLUMNS:", squads.columns)
print(squads.head(5))

pm = pl.read_parquet("../data/processed/player_metadata.parquet")
print("\nPLAYER_METADATA COLUMNS:", pm.columns)
print(pm.head(5))

SQUADS COLUMNS: ['id', 'name']
shape: (5, 2)
┌─────┬──────────────────────────┐
│ id  ┆ name                     │
│ --- ┆ ---                      │
│ i64 ┆ str                      │
╞═════╪══════════════════════════╡
│ 27  ┆ 1. FC Köln               │
│ 29  ┆ Borussia Dortmund        │
│ 30  ┆ VfL Wolfsburg            │
│ 31  ┆ TSG 1899 Hoffenheim      │
│ 32  ┆ Borussia Mönchengladbach │
└─────┴──────────────────────────┘

PLAYER_METADATA COLUMNS: ['player_id', 'team_id', 'total_events', 'matches_played']
shape: (5, 4)
┌───────────┬─────────┬──────────────┬────────────────┐
│ player_id ┆ team_id ┆ total_events ┆ matches_played │
│ ---       ┆ ---     ┆ ---          ┆ ---            │
│ str       ┆ str     ┆ u32          ┆ u32            │
╞═══════════╪═════════╪══════════════╪════════════════╡
│ 281       ┆ 41      ┆ 9375         ┆ 33             │
│ 32214     ┆ 29      ┆ 7816         ┆ 33             │
│ 1359      ┆ 46      ┆ 7802         ┆ 33             │
│ null      ┆ null    ┆

In [26]:
df_points = pl.read_parquet("../data/processed/all_events_with_points.parquet")
print("COLUMNS:", df_points.columns)
print("\nSHAPE:", df_points.shape)
print("\nEVENT TYPE SAMPLE:")
print(df_points.select(["player_id", "event_type", "result", "pass_type", "success"]).head(10))
print("\nEVENT TYPE COUNTS:")
print(df_points.group_by("event_type").len().sort("len", descending=True))

COLUMNS: ['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp', 'ball_state', 'ball_owning_team', 'team_id', 'player_id', 'coordinates_x', 'coordinates_y', 'end_coordinates_x', 'end_coordinates_y', 'receiver_player_id', 'body_part_type', 'set_piece_type', 'result', 'success', 'duel_type', 'is_under_pressure', 'pass_type', 'goalkeeper_type', 'match_id', 'zone', 'end_zone', 'card_type', 'attack_points', 'defense_points', 'passing_points']

SHAPE: (962990, 29)

EVENT TYPE SAMPLE:
shape: (10, 5)
┌───────────┬───────────────────┬────────────┬───────────┬─────────┐
│ player_id ┆ event_type        ┆ result     ┆ pass_type ┆ success │
│ ---       ┆ ---               ┆ ---        ┆ ---       ┆ ---     │
│ str       ┆ str               ┆ str        ┆ str       ┆ bool    │
╞═══════════╪═══════════════════╪════════════╪═══════════╪═════════╡
│ 204       ┆ PASS              ┆ INCOMPLETE ┆ null      ┆ false   │
│ null      ┆ GENERIC:NO_VIDEO  ┆ null       ┆ null      ┆ null    │
│ 120

In [27]:
# Cell: Compute raw stats per player and enrich the final parquet

print("Computing raw stats per player...")

raw_stats = (
    df_points
    .filter(pl.col("player_id").is_not_null())
    .with_columns([
        # Progressive pass flag: completed pass advancing ≥10m forward
        (
            (pl.col("event_type") == "PASS") &
            (pl.col("success") == True) &
            (pl.col("end_coordinates_x") - pl.col("coordinates_x") >= 10)
        ).alias("is_progressive_pass")
    ])
    .group_by("player_id")
    .agg([
        # Goals
        pl.col("event_type").filter(
            (pl.col("event_type") == "SHOT") & (pl.col("result") == "GOAL")
        ).len().alias("goals"),

        # Shots
        pl.col("event_type").filter(
            pl.col("event_type") == "SHOT"
        ).len().alias("shots"),

        # Assists (shot-assist passes)
        pl.col("pass_type").filter(
            pl.col("pass_type") == "SHOT_ASSIST"
        ).len().alias("assists"),

        # Duels won
        pl.col("event_type").filter(
            (pl.col("event_type") == "DUEL") & (pl.col("result") == "WON")
        ).len().alias("duels_won"),

        # Duels total
        pl.col("event_type").filter(
            pl.col("event_type") == "DUEL"
        ).len().alias("duels_total"),

        # Interceptions
        pl.col("event_type").filter(
            pl.col("event_type") == "INTERCEPTION"
        ).len().alias("interceptions"),

        # Recoveries
        pl.col("event_type").filter(
            pl.col("event_type") == "RECOVERY"
        ).len().alias("recoveries"),

        # Successful carries
        pl.col("event_type").filter(
            (pl.col("event_type") == "CARRY") & (pl.col("result") == "COMPLETE")
        ).len().alias("carries_completed"),

        # Progressive passes
        pl.col("is_progressive_pass").filter(
            pl.col("is_progressive_pass") == True
        ).len().alias("progressive_passes"),

        # Pass completion
        pl.col("event_type").filter(
            (pl.col("event_type") == "PASS") & (pl.col("success") == True)
        ).len().alias("passes_completed"),

        pl.col("event_type").filter(
            pl.col("event_type") == "PASS"
        ).len().alias("passes_total"),
    ])
    .with_columns([
        # Pass completion % — rounded to 1 decimal
        pl.when(pl.col("passes_total") > 0)
          .then((pl.col("passes_completed") / pl.col("passes_total") * 100).round(1))
          .otherwise(0.0)
          .alias("pass_completion_pct"),

        # Duel win % 
        pl.when(pl.col("duels_total") > 0)
          .then((pl.col("duels_won") / pl.col("duels_total") * 100).round(1))
          .otherwise(0.0)
          .alias("duel_win_pct"),

        # Shot conversion %
        pl.when(pl.col("shots") > 0)
          .then((pl.col("goals") / pl.col("shots") * 100).round(1))
          .otherwise(0.0)
          .alias("shot_conversion_pct"),
    ])
)

print(f" Computed stats for {len(raw_stats)} players")

# Quick sanity check — Kane should have ~36 goals
kane_check = raw_stats.filter(pl.col("player_id") == "204")  
# (204 is likely not Kane — just checking the structure)
print("\nSample (first 3 rows):")
print(raw_stats.head(3))

# Load existing final parquet and join
final = pl.read_parquet("../data/processed/player_ratings_FINAL_with_market_values.parquet")
print(f"\nExisting parquet: {len(final)} players, {len(final.columns)} columns")

# Join raw stats in
final_enriched = final.join(
    raw_stats,
    on="player_id",
    how="left"
)

print(f"Enriched parquet: {len(final_enriched)} players, {len(final_enriched.columns)} columns")
print("\nNew columns added:", [c for c in final_enriched.columns if c not in final.columns])

# Save — overwrite the existing final parquet
final_enriched.write_parquet(
    "../data/processed/player_ratings_FINAL_with_market_values.parquet"
)
print("\n Saved enriched parquet!")

# Sanity check on a known forward
print("\nTop 5 goal scorers in dataset:")
print(
    final_enriched
    .select(["player_name", "position", "goals", "shots", "assists", "shot_conversion_pct"])
    .sort("goals", descending=True)
    .head(5)
)

Computing raw stats per player...
 Computed stats for 493 players

Sample (first 3 rows):
shape: (3, 15)
┌───────────┬───────┬───────┬─────────┬───┬──────────────┬─────────────┬─────────────┬─────────────┐
│ player_id ┆ goals ┆ shots ┆ assists ┆ … ┆ passes_total ┆ pass_comple ┆ duel_win_pc ┆ shot_conver │
│ ---       ┆ ---   ┆ ---   ┆ ---     ┆   ┆ ---          ┆ tion_pct    ┆ t           ┆ sion_pct    │
│ str       ┆ u32   ┆ u32   ┆ u32     ┆   ┆ u32          ┆ ---         ┆ ---         ┆ ---         │
│           ┆       ┆       ┆         ┆   ┆              ┆ f64         ┆ f64         ┆ f64         │
╞═══════════╪═══════╪═══════╪═════════╪═══╪══════════════╪═════════════╪═════════════╪═════════════╡
│ 16806     ┆ 0     ┆ 13    ┆ 35      ┆ … ┆ 730          ┆ 61.0        ┆ 40.5        ┆ 0.0         │
│ 106573    ┆ 0     ┆ 0     ┆ 0       ┆ … ┆ 22           ┆ 36.4        ┆ 0.0         ┆ 0.0         │
│ 6371      ┆ 8     ┆ 46    ┆ 47      ┆ … ┆ 1004         ┆ 48.4        ┆ 43.2        ┆ 

In [28]:
df = pl.read_parquet("../data/processed/player_ratings_FINAL_with_market_values.parquet")
print([c for c in df.columns if 'avg' in c.lower() or 'y' in c.lower()])

# Also check season_avg_x range to calibrate thresholds
print(df.select(["player_name", "position", "season_avg_x"])
      .sort("season_avg_x", descending=True)
      .head(10))
print(df.select(["player_name", "position", "season_avg_x"])
      .sort("season_avg_x")
      .head(10))

['player_id', 'total_passing_accuracy', 'matches_played', 'player_name', 'season_avg_x', 'passing_accuracy_per90', 'passing_accuracy_rating', 'value_category']
shape: (10, 3)
┌─────────────────────────┬──────────┬──────────────┐
│ player_name             ┆ position ┆ season_avg_x │
│ ---                     ┆ ---      ┆ ---          │
│ str                     ┆ str      ┆ f64          │
╞═════════════════════════╪══════════╪══════════════╡
│ Borja Iglesias          ┆ forward  ┆ 21.930894    │
│ Kingsley Coman          ┆ forward  ┆ 18.561482    │
│ Loïs Openda             ┆ forward  ┆ 17.631121    │
│ Patrik Schick           ┆ forward  ┆ 17.274971    │
│ Victor Boniface         ┆ forward  ┆ 17.23027     │
│ Christopher Antwi-Adjei ┆ forward  ┆ 17.141927    │
│ Sheraldo Becker         ┆ forward  ┆ 17.001734    │
│ Maximilian Beier        ┆ forward  ┆ 16.817354    │
│ Justin Diehl            ┆ forward  ┆ 16.322066    │
│ Leroy Sané              ┆ forward  ┆ 16.257619    │
└──────────────

In [29]:
role_features = (
    pl.read_parquet("../data/processed/all_events_with_points.parquet")
    .filter(pl.col("player_id").is_not_null())
    .filter(~pl.col("event_type").str.starts_with("GENERIC"))
    .group_by("player_id")
    .agg([
        pl.col("coordinates_y").mean().alias("season_avg_y"),
        pl.col("coordinates_y").abs().mean().alias("season_avg_y_abs"),
        pl.len().alias("total_events"),
        pl.col("event_type").filter(pl.col("event_type") == "SHOT")
          .len().alias("n_shots"),
        pl.col("event_type").filter(pl.col("event_type") == "CARRY")
          .len().alias("n_carries"),
        pl.col("event_type").filter(pl.col("event_type") == "DUEL")
          .len().alias("n_duels"),
        pl.col("event_type").filter(pl.col("event_type") == "RECOVERY")
          .len().alias("n_recoveries"),
        pl.col("event_type").filter(pl.col("event_type") == "CLEARANCE")
          .len().alias("n_clearances"),
        pl.col("event_type").filter(pl.col("event_type") == "INTERCEPTION")
          .len().alias("n_interceptions"),
        pl.col("pass_type").filter(pl.col("pass_type") == "SHOT_ASSIST")
          .len().alias("n_shot_assists"),
    ])
)

print(role_features.head(5))
print("\nseason_avg_y range:")
print(role_features.select(pl.col("season_avg_y").min().alias("min"),
                            pl.col("season_avg_y").max().alias("max"),
                            pl.col("season_avg_y_abs").mean().alias("avg_abs_y")))

shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ player_id ┆ season_av ┆ season_av ┆ total_eve ┆ … ┆ n_recover ┆ n_clearan ┆ n_interce ┆ n_shot_a │
│ ---       ┆ g_y       ┆ g_y_abs   ┆ nts       ┆   ┆ ies       ┆ ces       ┆ ptions    ┆ ssists   │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ u32       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 64020     ┆ -4.789655 ┆ 21.439554 ┆ 992       ┆ … ┆ 45        ┆ 14        ┆ 27        ┆ 13       │
│ 715       ┆ 0.581631  ┆ 16.15922  ┆ 1446      ┆ … ┆ 64        ┆ 32        ┆ 38        ┆ 14       │
│ 10035     ┆ 11.93208  ┆ 15.471297 ┆ 3942      ┆ … ┆ 442       ┆ 105       ┆ 107       ┆ 2        │
│ 3153      ┆ -7.151218 ┆ 17.318743 ┆ 2884      ┆ … ┆ 169       ┆ 48        

In [30]:
# ── Cell: Derive granular roles and save to enriched parquet ──────────────────

import polars as pl
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# Load features computed in previous cell
role_features = (
    pl.read_parquet(PROCESSED_DIR / "all_events_with_points.parquet")
    .filter(pl.col("player_id").is_not_null())
    .filter(~pl.col("event_type").str.starts_with("GENERIC"))
    .group_by("player_id")
    .agg([
        pl.col("coordinates_y").mean().alias("season_avg_y"),
        pl.col("coordinates_y").abs().mean().alias("season_avg_y_abs"),
        pl.len().alias("total_events"),
        pl.col("event_type").filter(pl.col("event_type") == "SHOT")
          .len().alias("n_shots"),
        pl.col("event_type").filter(pl.col("event_type") == "CARRY")
          .len().alias("n_carries"),
        pl.col("event_type").filter(pl.col("event_type") == "DUEL")
          .len().alias("n_duels"),
        pl.col("event_type").filter(pl.col("event_type") == "RECOVERY")
          .len().alias("n_recoveries"),
        pl.col("event_type").filter(pl.col("event_type") == "CLEARANCE")
          .len().alias("n_clearances"),
        pl.col("event_type").filter(pl.col("event_type") == "INTERCEPTION")
          .len().alias("n_interceptions"),
        pl.col("pass_type").filter(pl.col("pass_type") == "SHOT_ASSIST")
          .len().alias("n_shot_assists"),
    ])
    # Compute ratios (avoid div by zero)
    .with_columns([
        (pl.col("n_shots")        / pl.col("total_events")).alias("shot_ratio"),
        (pl.col("n_carries")      / pl.col("total_events")).alias("carry_ratio"),
        (pl.col("n_duels")        / pl.col("total_events")).alias("duel_ratio"),
        (pl.col("n_recoveries")   / pl.col("total_events")).alias("recovery_ratio"),
        (pl.col("n_clearances")   / pl.col("total_events")).alias("clearance_ratio"),
        (pl.col("n_shot_assists") / pl.col("total_events")).alias("shot_assist_ratio"),
    ])
)

# Join with main parquet to get season_avg_x and broad position
final = pl.read_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
)

# Force both sides to string to prevent silent type mismatch
role_features_cast = role_features.with_columns(
    pl.col("player_id").cast(pl.Utf8)
)
final_cast = final.with_columns(
    pl.col("player_id").cast(pl.Utf8)
)

df = final_cast.join(
    role_features_cast.select([
        "player_id", "season_avg_y", "season_avg_y_abs",
        "shot_ratio", "carry_ratio", "duel_ratio",
        "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
    ]),
    on="player_id", how="left"
)

# ── Role classification function ──────────────────────────────────────────────
# Thresholds calibrated from data:
#   x:  forwards ~16-22, mids ~-5 to 15, defenders ~-15 to -30, GKs ~< -38
#   y_abs: central < 12, half-wide 12-20, wide > 20

def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, recovery_r, clearance_r, shot_assist_r):

    # Handle nulls
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"  # negative y = left in secondspectrum

    # ── Goalkeeper ────────────────────────────────────────────────────────────
    if avg_x < -38:
        return "GK"

    # ── Defenders ────────────────────────────────────────────────────────────
    if avg_x < -15:
        if avg_y_abs > 19:
            return f"{side}B"          # LB or RB
        else:
            return "CB"

    # ── Defensive / Central Midfield ─────────────────────────────────────────
    if avg_x < 2:
        if avg_y_abs > 19:
            return f"{side}M"
        elif recovery_r > 0.08 or clearance_r > 0.03:
            return "CDM"
        else:
            return "CM"

    # ── Central / Attacking Midfield ─────────────────────────────────────────
    if avg_x < 14:
        if avg_y_abs > 19:
            return f"{side}M"
        elif shot_assist_r > 0.015 or avg_x > 8:
            return "CAM"
        else:
            return "CM"

    # ── Attacking third ───────────────────────────────────────────────────────
    # Tightened: only classify as winger if genuinely wide (y_abs > 20)
    # AND carries more than shots — pure strikers stay central even if slightly wide
    if avg_y_abs > 20 and carry_r > shot_r * 1.5:
        return f"{side}W"              # True winger — wide + carry-heavy
    elif avg_y_abs > 15 and carry_r > shot_r * 2.0:
        return f"{side}W"              # Wide and very carry-dominant
    elif shot_r > 0.035:
        return "ST"                    # Central + shot-heavy = striker
    else:
        return "CF"                    # Central forward, more creative/mobile

# Apply classification
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs"),
        row.get("shot_ratio",      0) or 0,
        row.get("carry_ratio",     0) or 0,
        row.get("recovery_ratio",  0) or 0,
        row.get("clearance_ratio", 0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

# ── Sanity check ──────────────────────────────────────────────────────────────
print("ROLE DISTRIBUTION:")
print(df.group_by("role").len().sort("len", descending=True))

print("\nSAMPLE — Forwards:")
print(df.filter(pl.col("position") == "forward")
      .select(["player_name", "role", "season_avg_x", "season_avg_y"])
      .sort("season_avg_x", descending=True).head(15))

print("\nSAMPLE — Defenders:")
print(df.filter(pl.col("position") == "defender")
      .select(["player_name", "role", "season_avg_x", "season_avg_y"])
      .sort("season_avg_x").head(15))

print("\nSAMPLE — Midfielders:")
print(df.filter(pl.col("position") == "midfielder")
      .select(["player_name", "role", "season_avg_x", "season_avg_y"])
      .sort("season_avg_x", descending=True).head(15))

# ── Save ──────────────────────────────────────────────────────────────────────
# Drop helper columns we don't need in the dashboard
df_save = df.drop([
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
])
df_save.write_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
)
print(f"\n Saved with 'role' column. Total players: {len(df_save)}")
print("New columns:", [c for c in df_save.columns if c not in final.columns])

ROLE DISTRIBUTION:
shape: (11, 2)
┌──────┬─────┐
│ role ┆ len │
│ ---  ┆ --- │
│ str  ┆ u32 │
╞══════╪═════╡
│ CM   ┆ 72  │
│ RM   ┆ 67  │
│ LM   ┆ 59  │
│ CAM  ┆ 54  │
│ CDM  ┆ 48  │
│ …    ┆ …   │
│ GK   ┆ 21  │
│ RW   ┆ 17  │
│ LW   ┆ 10  │
│ ST   ┆ 4   │
│ LB   ┆ 1   │
└──────┴─────┘

SAMPLE — Forwards:
shape: (15, 4)
┌─────────────────┬──────┬──────────────┬──────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y │
│ ---             ┆ ---  ┆ ---          ┆ ---          │
│ str             ┆ str  ┆ f64          ┆ f64          │
╞═════════════════╪══════╪══════════════╪══════════════╡
│ Borja Iglesias  ┆ RW   ┆ 21.930894    ┆ 1.364198     │
│ Kingsley Coman  ┆ LW   ┆ 18.561482    ┆ -3.742539    │
│ Loïs Openda     ┆ RW   ┆ 17.631121    ┆ 0.181242     │
│ Patrik Schick   ┆ ST   ┆ 17.274971    ┆ -0.190317    │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6.59159      │
│ …               ┆ …    ┆ …            ┆ …            │
│ Thomas Müller   ┆ LW   ┆ 16.052338    ┆ -11.536

In [31]:
# ── Cell: Derive granular roles and save to enriched parquet ──────────────────

import polars as pl
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# ── Step 1: Compute features from events ─────────────────────────────────────
role_features = (
    pl.read_parquet(PROCESSED_DIR / "all_events_with_points.parquet")
    .filter(pl.col("player_id").is_not_null())
    .filter(~pl.col("event_type").str.starts_with("GENERIC"))
    .group_by("player_id")
    .agg([
        pl.col("coordinates_y").mean().alias("season_avg_y"),
        pl.col("coordinates_y").abs().mean().alias("season_avg_y_abs"),
        pl.len().alias("total_events"),
        pl.col("event_type").filter(pl.col("event_type") == "SHOT")
          .len().alias("n_shots"),
        pl.col("event_type").filter(pl.col("event_type") == "CARRY")
          .len().alias("n_carries"),
        pl.col("event_type").filter(pl.col("event_type") == "DUEL")
          .len().alias("n_duels"),
        pl.col("event_type").filter(pl.col("event_type") == "RECOVERY")
          .len().alias("n_recoveries"),
        pl.col("event_type").filter(pl.col("event_type") == "CLEARANCE")
          .len().alias("n_clearances"),
        pl.col("event_type").filter(pl.col("event_type") == "INTERCEPTION")
          .len().alias("n_interceptions"),
        pl.col("pass_type").filter(pl.col("pass_type") == "SHOT_ASSIST")
          .len().alias("n_shot_assists"),
    ])
    .with_columns([
        (pl.col("n_shots")        / pl.col("total_events")).alias("shot_ratio"),
        (pl.col("n_carries")      / pl.col("total_events")).alias("carry_ratio"),
        (pl.col("n_duels")        / pl.col("total_events")).alias("duel_ratio"),
        (pl.col("n_recoveries")   / pl.col("total_events")).alias("recovery_ratio"),
        (pl.col("n_clearances")   / pl.col("total_events")).alias("clearance_ratio"),
        (pl.col("n_shot_assists") / pl.col("total_events")).alias("shot_assist_ratio"),
    ])
    .with_columns(pl.col("player_id").cast(pl.Utf8))
)

# ── Step 2: Load parquet and DROP stale spatial/role columns before joining ───
final = pl.read_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
).with_columns(pl.col("player_id").cast(pl.Utf8))

# Drop any columns that might already exist from a previous run
cols_to_drop = [c for c in final.columns if c in [
    "season_avg_y", "season_avg_y_abs", "season_avg_y_right",
    "season_avg_y_abs_right", "role",
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
]]
if cols_to_drop:
    print(f"Dropping stale columns: {cols_to_drop}")
    final = final.drop(cols_to_drop)

df = final.join(
    role_features.select([
        "player_id", "season_avg_y", "season_avg_y_abs",
        "shot_ratio", "carry_ratio", "duel_ratio",
        "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
    ]),
    on="player_id", how="left"
)

# ── Step 3: Classification function ──────────────────────────────────────────
# Key fix: winger classification now requires BOTH genuinely wide avg_y
# (not just avg_y_abs) AND carry dominance. Central forwards with slight
# lateral drift (Iglesias avg_y=1.4) stay as ST/CF.

def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, recovery_r, clearance_r, shot_assist_r):
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"

    # ── Goalkeeper ────────────────────────────────────────────────────────────
    if avg_x < -38:
        return "GK"

    # ── Defenders ────────────────────────────────────────────────────────────
    if avg_x < -15:
        return f"{side}B" if avg_y_abs > 19 else "CB"

    # ── Defensive / Central Midfield ─────────────────────────────────────────
    if avg_x < 2:
        if avg_y_abs > 19:
            return f"{side}M"
        return "CDM" if (recovery_r > 0.08 or clearance_r > 0.03) else "CM"

    # ── Central / Attacking Midfield ─────────────────────────────────────────
    if avg_x < 14:
        if avg_y_abs > 19:
            return f"{side}M"
        return "CAM" if (shot_assist_r > 0.015 or avg_x > 8) else "CM"

    # ── Attacking third ───────────────────────────────────────────────────────
    # Winger: must be BOTH wide in absolute terms (avg_y_abs > 20)
    # AND meaningfully off-centre (abs avg_y > 12 so Iglesias at 1.4 is excluded)
    # AND carry-heavy relative to shots
    is_wide     = avg_y_abs > 20 and abs(avg_y) > 12
    carry_heavy = carry_r > shot_r * 2.0

    if is_wide and carry_heavy:
        return f"{side}W"
    elif shot_r > 0.03:
        return "ST"
    else:
        return "CF"

# ── Step 4: Apply classification ─────────────────────────────────────────────
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs") or 0,
        row.get("shot_ratio",        0) or 0,
        row.get("carry_ratio",       0) or 0,
        row.get("recovery_ratio",    0) or 0,
        row.get("clearance_ratio",   0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

# ── Step 5: Sanity checks ─────────────────────────────────────────────────────
print("ROLE DISTRIBUTION:")
print(df.group_by("role").len().sort("len", descending=True))

print("\nKEY PLAYER CHECKS:")
key_players = ["Harry Kane", "Leroy Sané", "Kingsley Coman", "Loïs Openda",
               "Victor Boniface", "Benjamin Sesko", "Joshua Kimmich",
               "Granit Xhaka", "Florian Wirtz", "Dayot Upamecano"]
check = df.filter(pl.col("player_name").is_in(key_players))
print(check.select(["player_name", "role", "season_avg_x", "season_avg_y",
                    "season_avg_y_abs"]).sort("season_avg_x", descending=True))

# ── Step 6: Save — drop helper ratio columns, keep spatial ones ───────────────
df_save = df.drop([
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
])
df_save.write_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
)
print(f"\n Saved. Players: {len(df_save)}")
print("Added columns:", [c for c in df_save.columns if c not in final.columns])


Dropping stale columns: ['season_avg_y', 'season_avg_y_abs', 'role']
ROLE DISTRIBUTION:
shape: (12, 2)
┌──────┬─────┐
│ role ┆ len │
│ ---  ┆ --- │
│ str  ┆ u32 │
╞══════╪═════╡
│ CM   ┆ 72  │
│ RM   ┆ 67  │
│ LM   ┆ 59  │
│ CAM  ┆ 54  │
│ CDM  ┆ 48  │
│ …    ┆ …   │
│ ST   ┆ 17  │
│ CF   ┆ 10  │
│ RW   ┆ 2   │
│ LW   ┆ 2   │
│ LB   ┆ 1   │
└──────┴─────┘

KEY PLAYER CHECKS:
shape: (10, 5)
┌─────────────────┬──────┬──────────────┬──────────────┬──────────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y ┆ season_avg_y_abs │
│ ---             ┆ ---  ┆ ---          ┆ ---          ┆ ---              │
│ str             ┆ str  ┆ f64          ┆ f64          ┆ f64              │
╞═════════════════╪══════╪══════════════╪══════════════╪══════════════════╡
│ Kingsley Coman  ┆ CF   ┆ 18.561482    ┆ -3.742539    ┆ 23.581972        │
│ Loïs Openda     ┆ ST   ┆ 17.631121    ┆ 0.181242     ┆ 16.39865         │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6.59159      ┆ 14.136506        │

In [32]:
# ── Cell: Derive granular roles and save to enriched parquet ──────────────────

import polars as pl
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# ── Step 1: Compute features from events ─────────────────────────────────────
role_features = (
    pl.read_parquet(PROCESSED_DIR / "all_events_with_points.parquet")
    .filter(pl.col("player_id").is_not_null())
    .filter(~pl.col("event_type").str.starts_with("GENERIC"))
    .group_by("player_id")
    .agg([
        pl.col("coordinates_y").mean().alias("season_avg_y"),
        pl.col("coordinates_y").abs().mean().alias("season_avg_y_abs"),
        pl.len().alias("total_events"),
        pl.col("event_type").filter(pl.col("event_type") == "SHOT")
          .len().alias("n_shots"),
        pl.col("event_type").filter(pl.col("event_type") == "CARRY")
          .len().alias("n_carries"),
        pl.col("event_type").filter(pl.col("event_type") == "DUEL")
          .len().alias("n_duels"),
        pl.col("event_type").filter(pl.col("event_type") == "RECOVERY")
          .len().alias("n_recoveries"),
        pl.col("event_type").filter(pl.col("event_type") == "CLEARANCE")
          .len().alias("n_clearances"),
        pl.col("event_type").filter(pl.col("event_type") == "INTERCEPTION")
          .len().alias("n_interceptions"),
        pl.col("pass_type").filter(pl.col("pass_type") == "SHOT_ASSIST")
          .len().alias("n_shot_assists"),
    ])
    .with_columns([
        (pl.col("n_shots")        / pl.col("total_events")).alias("shot_ratio"),
        (pl.col("n_carries")      / pl.col("total_events")).alias("carry_ratio"),
        (pl.col("n_duels")        / pl.col("total_events")).alias("duel_ratio"),
        (pl.col("n_recoveries")   / pl.col("total_events")).alias("recovery_ratio"),
        (pl.col("n_clearances")   / pl.col("total_events")).alias("clearance_ratio"),
        (pl.col("n_shot_assists") / pl.col("total_events")).alias("shot_assist_ratio"),
    ])
    .with_columns(pl.col("player_id").cast(pl.Utf8))
)

# ── Step 2: Load parquet and DROP stale spatial/role columns before joining ───
final = pl.read_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
).with_columns(pl.col("player_id").cast(pl.Utf8))

# Drop any columns that might already exist from a previous run
cols_to_drop = [c for c in final.columns if c in [
    "season_avg_y", "season_avg_y_abs", "season_avg_y_right",
    "season_avg_y_abs_right", "role",
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
]]
if cols_to_drop:
    print(f"Dropping stale columns: {cols_to_drop}")
    final = final.drop(cols_to_drop)

df = final.join(
    role_features.select([
        "player_id", "season_avg_y", "season_avg_y_abs",
        "shot_ratio", "carry_ratio", "duel_ratio",
        "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
    ]),
    on="player_id", how="left"
)

# ── Step 3: Classification function ──────────────────────────────────────────
# Key fix: winger classification now requires BOTH genuinely wide avg_y
# (not just avg_y_abs) AND carry dominance. Central forwards with slight
# lateral drift (Iglesias avg_y=1.4) stay as ST/CF.

def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, duel_ratio, recovery_r, clearance_r, shot_assist_r):
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"

    # ── Goalkeeper ────────────────────────────────────────────────────────────
    if avg_x < -38:
        return "GK"

    # ── Defenders ────────────────────────────────────────────────────────────
    # Extended upper bound to -8 to catch deep CBs like Upamecano (avg_x=-10.5)
    # who get misclassified as CM due to high defensive avg_x
    if avg_x < -8:
        if avg_y_abs > 19:
            return f"{side}B"
        # Extra CB signal: high clearance or duel ratio even if not super deep
        if clearance_r > 0.02 or duel_ratio > 0.06:
            return "CB"
        return "CB" if avg_y_abs < 17 else f"{side}B"

    # ── Defensive / Central Midfield ─────────────────────────────────────────
    if avg_x < 2:
        if avg_y_abs > 19:
            return f"{side}M"
        # Kimmich fix: wide avg_y_abs + progressive = RB not CM
        if avg_y_abs > 14 and carry_r > 0.18:
            return f"{side}B"
        return "CDM" if (recovery_r > 0.08 or clearance_r > 0.03) else "CM"

    # ── Central / Attacking Midfield ─────────────────────────────────────────
    if avg_x < 14:
        if avg_y_abs > 19:
            return f"{side}M"
        return "CAM" if (shot_assist_r > 0.015 or avg_x > 8) else "CM"

    # ── Attacking third ───────────────────────────────────────────────────────
    # Winger: must be BOTH wide in absolute terms (avg_y_abs > 20)
    # AND meaningfully off-centre (abs avg_y > 12 so Iglesias at 1.4 is excluded)
    # AND carry-heavy relative to shots
    is_wide     = avg_y_abs > 20 and abs(avg_y) > 12
    carry_heavy = carry_r > shot_r * 2.0

    if is_wide and carry_heavy:
        return f"{side}W"
    elif shot_r > 0.03:
        return "ST"
    else:
        return "CF"

# ── Step 4: Apply classification ─────────────────────────────────────────────
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs") or 0,
        row.get("shot_ratio",        0) or 0,
        row.get("carry_ratio",       0) or 0,
        row.get("duel_ratio",        0) or 0,
        row.get("recovery_ratio",    0) or 0,
        row.get("clearance_ratio",   0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

# ── Step 5: Sanity checks ─────────────────────────────────────────────────────
print("ROLE DISTRIBUTION:")
print(df.group_by("role").len().sort("len", descending=True))

print("\nKEY PLAYER CHECKS:")
key_players = ["Harry Kane", "Leroy Sané", "Kingsley Coman", "Loïs Openda",
               "Victor Boniface", "Benjamin Sesko", "Joshua Kimmich",
               "Granit Xhaka", "Florian Wirtz", "Dayot Upamecano"]
check = df.filter(pl.col("player_name").is_in(key_players))
print(check.select(["player_name", "role", "season_avg_x", "season_avg_y",
                    "season_avg_y_abs"]).sort("season_avg_x", descending=True))

# ── Step 6: Save — drop helper ratio columns, keep spatial ones ───────────────
df_save = df.drop([
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
])
df_save.write_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
)
print(f"\n Saved. Players: {len(df_save)}")
print("Added columns:", [c for c in df_save.columns if c not in final.columns])

Dropping stale columns: ['season_avg_y', 'season_avg_y_abs', 'role']
ROLE DISTRIBUTION:
shape: (13, 2)
┌──────┬─────┐
│ role ┆ len │
│ ---  ┆ --- │
│ str  ┆ u32 │
╞══════╪═════╡
│ CB   ┆ 81  │
│ RM   ┆ 66  │
│ LM   ┆ 55  │
│ CAM  ┆ 54  │
│ CM   ┆ 39  │
│ …    ┆ …   │
│ ST   ┆ 17  │
│ CF   ┆ 10  │
│ CDM  ┆ 7   │
│ RW   ┆ 2   │
│ LW   ┆ 2   │
└──────┴─────┘

KEY PLAYER CHECKS:
shape: (10, 5)
┌─────────────────┬──────┬──────────────┬──────────────┬──────────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y ┆ season_avg_y_abs │
│ ---             ┆ ---  ┆ ---          ┆ ---          ┆ ---              │
│ str             ┆ str  ┆ f64          ┆ f64          ┆ f64              │
╞═════════════════╪══════╪══════════════╪══════════════╪══════════════════╡
│ Kingsley Coman  ┆ CF   ┆ 18.561482    ┆ -3.742539    ┆ 23.581972        │
│ Loïs Openda     ┆ ST   ┆ 17.631121    ┆ 0.181242     ┆ 16.39865         │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6.59159      ┆ 14.136506        │

In [33]:
# Temporary debug cell — run this before the roles loop
sane = df.filter(pl.col("player_name") == "Leroy Sané").to_dicts()[0]
print("Sané debug:")
print(f"  avg_x       = {sane.get('season_avg_x')}")
print(f"  avg_y       = {sane.get('season_avg_y')}")  
print(f"  avg_y_abs   = {sane.get('season_avg_y_abs')}")
print(f"  shot_ratio  = {sane.get('shot_ratio')}")
print(f"  carry_ratio = {sane.get('carry_ratio')}")
print(f"  is_wide     = {sane.get('season_avg_y_abs', 0) > 20 and sane.get('season_avg_y_abs', 0) > 12}")
print(f"  carry_heavy = {sane.get('carry_ratio', 0) > sane.get('shot_ratio', 0) * 2.0}")

coman = df.filter(pl.col("player_name") == "Kingsley Coman").to_dicts()[0]
print("\nComan debug:")
print(f"  avg_y_abs   = {coman.get('season_avg_y_abs')}")
print(f"  shot_ratio  = {coman.get('shot_ratio')}")
print(f"  carry_ratio = {coman.get('carry_ratio')}")
print(f"  carry_heavy = {coman.get('carry_ratio', 0) > coman.get('shot_ratio', 0) * 2.0}")

Sané debug:
  avg_x       = 16.257618787547788
  avg_y       = -7.700718849840256
  avg_y_abs   = 20.215575079872206
  shot_ratio  = 0.026388341866876722
  carry_ratio = 0.3607719574635683
  is_wide     = True
  carry_heavy = True

Coman debug:
  avg_y_abs   = 23.581971640783255
  shot_ratio  = 0.020666666666666667
  carry_ratio = 0.394
  carry_heavy = True


In [34]:
# ── Cell: Derive granular roles and save to enriched parquet ──────────────────

import polars as pl
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# ── Step 1: Compute features from events ─────────────────────────────────────
role_features = (
    pl.read_parquet(PROCESSED_DIR / "all_events_with_points.parquet")
    .filter(pl.col("player_id").is_not_null())
    .filter(~pl.col("event_type").str.starts_with("GENERIC"))
    .group_by("player_id")
    .agg([
        pl.col("coordinates_y").mean().alias("season_avg_y"),
        pl.col("coordinates_y").abs().mean().alias("season_avg_y_abs"),
        pl.len().alias("total_events"),
        pl.col("event_type").filter(pl.col("event_type") == "SHOT")
          .len().alias("n_shots"),
        pl.col("event_type").filter(pl.col("event_type") == "CARRY")
          .len().alias("n_carries"),
        pl.col("event_type").filter(pl.col("event_type") == "DUEL")
          .len().alias("n_duels"),
        pl.col("event_type").filter(pl.col("event_type") == "RECOVERY")
          .len().alias("n_recoveries"),
        pl.col("event_type").filter(pl.col("event_type") == "CLEARANCE")
          .len().alias("n_clearances"),
        pl.col("event_type").filter(pl.col("event_type") == "INTERCEPTION")
          .len().alias("n_interceptions"),
        pl.col("pass_type").filter(pl.col("pass_type") == "SHOT_ASSIST")
          .len().alias("n_shot_assists"),
    ])
    .with_columns([
        (pl.col("n_shots")        / pl.col("total_events")).alias("shot_ratio"),
        (pl.col("n_carries")      / pl.col("total_events")).alias("carry_ratio"),
        (pl.col("n_duels")        / pl.col("total_events")).alias("duel_ratio"),
        (pl.col("n_recoveries")   / pl.col("total_events")).alias("recovery_ratio"),
        (pl.col("n_clearances")   / pl.col("total_events")).alias("clearance_ratio"),
        (pl.col("n_shot_assists") / pl.col("total_events")).alias("shot_assist_ratio"),
    ])
    .with_columns(pl.col("player_id").cast(pl.Utf8))
)

# ── Step 2: Load parquet and DROP stale spatial/role columns before joining ───
final = pl.read_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
).with_columns(pl.col("player_id").cast(pl.Utf8))

# Drop any columns that might already exist from a previous run
cols_to_drop = [c for c in final.columns if c in [
    "season_avg_y", "season_avg_y_abs", "season_avg_y_right",
    "season_avg_y_abs_right", "role",
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
]]
if cols_to_drop:
    print(f"Dropping stale columns: {cols_to_drop}")
    final = final.drop(cols_to_drop)

df = final.join(
    role_features.select([
        "player_id", "season_avg_y", "season_avg_y_abs",
        "shot_ratio", "carry_ratio", "duel_ratio",
        "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
    ]),
    on="player_id", how="left"
)

# ── Step 3: Classification function ──────────────────────────────────────────
# Key fix: winger classification now requires BOTH genuinely wide avg_y
# (not just avg_y_abs) AND carry dominance. Central forwards with slight
# lateral drift (Iglesias avg_y=1.4) stay as ST/CF.

def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, duel_ratio, recovery_r, clearance_r, shot_assist_r):
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"

    # ── Goalkeeper ────────────────────────────────────────────────────────────
    if avg_x < -38:
        return "GK"

    # ── Defenders ────────────────────────────────────────────────────────────
    # Extended upper bound to -8 to catch deep CBs like Upamecano (avg_x=-10.5)
    # who get misclassified as CM due to high defensive avg_x
    if avg_x < -8:
        if avg_y_abs > 19:
            return f"{side}B"
        # Extra CB signal: high clearance or duel ratio even if not super deep
        if clearance_r > 0.02 or duel_ratio > 0.06:
            return "CB"
        return "CB" if avg_y_abs < 17 else f"{side}B"

    # ── Defensive / Central Midfield ─────────────────────────────────────────
    if avg_x < 2:
        if avg_y_abs > 19:
            return f"{side}M"
        # Kimmich fix: wide avg_y_abs + progressive = RB not CM
        if avg_y_abs > 14 and carry_r > 0.18:
            return f"{side}B"
        return "CDM" if (recovery_r > 0.08 or clearance_r > 0.03) else "CM"

    # ── Central / Attacking Midfield ─────────────────────────────────────────
    if avg_x < 14:
        if avg_y_abs > 19:
            return f"{side}M"
        return "CAM" if (shot_assist_r > 0.015 or avg_x > 8) else "CM"

    # ── Attacking third ───────────────────────────────────────────────────────
    # Winger: must be BOTH wide in absolute terms (avg_y_abs > 20)
    # AND meaningfully off-centre (abs avg_y > 12 so Iglesias at 1.4 is excluded)
    # AND carry-heavy relative to shots
    is_wide     = avg_y_abs > 20 and abs(avg_y) > 12
    carry_heavy = carry_r > shot_r * 2.0

    if is_wide and carry_heavy:
        return f"{side}W"
    elif shot_r > 0.03:
        return "ST"
    else:
        return "CF"

# ── Step 4: Apply classification ─────────────────────────────────────────────
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs") or 0,
        row.get("shot_ratio",        0) or 0,
        row.get("carry_ratio",       0) or 0,
        row.get("duel_ratio",        0) or 0,
        row.get("recovery_ratio",    0) or 0,
        row.get("clearance_ratio",   0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

# ── Step 5: Sanity checks ─────────────────────────────────────────────────────
print("ROLE DISTRIBUTION:")
print(df.group_by("role").len().sort("len", descending=True))

print("\nKEY PLAYER CHECKS:")
key_players = ["Harry Kane", "Leroy Sané", "Kingsley Coman", "Loïs Openda",
               "Victor Boniface", "Benjamin Sesko", "Joshua Kimmich",
               "Granit Xhaka", "Florian Wirtz", "Dayot Upamecano"]
check = df.filter(pl.col("player_name").is_in(key_players))
print(check.select(["player_name", "role", "season_avg_x", "season_avg_y",
                    "season_avg_y_abs"]).sort("season_avg_x", descending=True))

# ── Step 6: Save — drop helper ratio columns, keep spatial ones ───────────────
df_save = df.drop([
    "shot_ratio", "carry_ratio", "duel_ratio",
    "recovery_ratio", "clearance_ratio", "shot_assist_ratio"
])
df_save.write_parquet(
    PROCESSED_DIR / "player_ratings_FINAL_with_market_values.parquet"
)
print(f"\n Saved. Players: {len(df_save)}")
print("Added columns:", [c for c in df_save.columns if c not in final.columns])

Dropping stale columns: ['season_avg_y', 'season_avg_y_abs', 'role']
ROLE DISTRIBUTION:
shape: (13, 2)
┌──────┬─────┐
│ role ┆ len │
│ ---  ┆ --- │
│ str  ┆ u32 │
╞══════╪═════╡
│ CB   ┆ 81  │
│ RM   ┆ 66  │
│ LM   ┆ 55  │
│ CAM  ┆ 54  │
│ CM   ┆ 39  │
│ …    ┆ …   │
│ ST   ┆ 17  │
│ CF   ┆ 10  │
│ CDM  ┆ 7   │
│ RW   ┆ 2   │
│ LW   ┆ 2   │
└──────┴─────┘

KEY PLAYER CHECKS:
shape: (10, 5)
┌─────────────────┬──────┬──────────────┬──────────────┬──────────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y ┆ season_avg_y_abs │
│ ---             ┆ ---  ┆ ---          ┆ ---          ┆ ---              │
│ str             ┆ str  ┆ f64          ┆ f64          ┆ f64              │
╞═════════════════╪══════╪══════════════╪══════════════╪══════════════════╡
│ Kingsley Coman  ┆ CF   ┆ 18.561482    ┆ -3.742539    ┆ 23.581972        │
│ Loïs Openda     ┆ ST   ┆ 17.631121    ┆ 0.181242     ┆ 16.39865         │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6.59159      ┆ 14.136506        │

In [35]:
# Test Sané directly against the NEW function
result = classify_role(
    avg_x=16.257,
    avg_y=-7.700,
    avg_y_abs=20.215,
    shot_r=0.02638,
    carry_r=0.36077,
    duel_ratio=0.05,
    recovery_r=0.04,
    clearance_r=0.01,
    shot_assist_r=0.01
)
print(f"Sané → {result}")  # Should print: Sané → LW

result2 = classify_role(
    avg_x=18.561,
    avg_y=-3.742,
    avg_y_abs=23.581,
    shot_r=0.02066,
    carry_r=0.394,
    duel_ratio=0.05,
    recovery_r=0.03,
    clearance_r=0.01,
    shot_assist_r=0.01
)
print(f"Coman → {result2}")  # Should print: Coman → LW

Sané → CF
Coman → CF


In [36]:
avg_x, avg_y, avg_y_abs = 16.257, -7.700, 20.215
shot_r, carry_r, duel_ratio = 0.02638, 0.36077, 0.05
recovery_r, clearance_r, shot_assist_r = 0.04, 0.01, 0.01

side = "L" if avg_y < 0 else "R"
print(f"side = {side}")
print(f"Block 1 (GK):      {avg_x < -38}")
print(f"Block 2 (DEF):     {avg_x < -8}")
print(f"Block 3 (DEF/MID): {avg_x < 2}")
print(f"Block 4 (MID):     {avg_x < 14}")
print(f"Block 5 (ATT):     {avg_x >= 14}")
print(f"  is_wide     = {avg_y_abs > 20}")
print(f"  carry_heavy = {carry_r > shot_r * 2.0}")
print(f"  shot_r > 0.03 = {shot_r > 0.03}")

import inspect
src = inspect.getsource(classify_role)
print(src[-300:])  # print last 300 chars to confirm it ends at block 5

side = L
Block 1 (GK):      False
Block 2 (DEF):     False
Block 3 (DEF/MID): False
Block 4 (MID):     False
Block 5 (ATT):     True
  is_wide     = True
  carry_heavy = True
  shot_r > 0.03 = False
 so Iglesias at 1.4 is excluded)
    # AND carry-heavy relative to shots
    is_wide     = avg_y_abs > 20 and abs(avg_y) > 12
    carry_heavy = carry_r > shot_r * 2.0

    if is_wide and carry_heavy:
        return f"{side}W"
    elif shot_r > 0.03:
        return "ST"
    else:
        return "CF"



In [37]:
def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, duel_ratio, recovery_r, clearance_r, shot_assist_r):
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"

    # 1. Goalkeeper
    if avg_x < -38:
        return "GK"

    # 2. Defenders
    if avg_x < -8:
        if avg_y_abs > 19:
            return f"{side}B"
        if clearance_r > 0.02 or duel_ratio > 0.06:
            return "CB"
        return "CB" if avg_y_abs < 17 else f"{side}B"

    # 3. Defensive / Central mid
    if avg_x < 2:
        if avg_y_abs > 19:
            return f"{side}M"
        if avg_y_abs > 14 and carry_r > 0.18:
            return f"{side}B"
        return "CDM" if (recovery_r > 0.08 or clearance_r > 0.03) else "CM"

    # 4. Central / Attacking mid
    if avg_x < 14:
        if avg_y_abs > 19:
            return f"{side}M"
        return "CAM" if (shot_assist_r > 0.015 or avg_x > 8) else "CM"

    # 5. Attacking third (avg_x >= 14)
    is_wide     = avg_y_abs > 20          # use avg_y_abs not abs(avg_y)
    carry_heavy = carry_r > shot_r * 2.0

    if is_wide and carry_heavy:
        return f"{side}W"
    elif shot_r > 0.03:
        return "ST"
    else:
        return "CF"

# Quick verify
print(classify_role(16.257, -7.700, 20.215, 0.02638, 0.36077, 0.05, 0.04, 0.01, 0.01))  # → LW
print(classify_role(18.561, -3.742, 23.581, 0.02066, 0.394,   0.05, 0.03, 0.01, 0.01))  # → LW
print(classify_role(15.252,  0.509, 13.775, 0.065,   0.20,    0.06, 0.04, 0.02, 0.008)) # → ST

LW
LW
ST


In [38]:
# Re-apply and save
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs") or 0,
        row.get("shot_ratio",        0) or 0,
        row.get("carry_ratio",       0) or 0,
        row.get("duel_ratio",        0) or 0,
        row.get("recovery_ratio",    0) or 0,
        row.get("clearance_ratio",   0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

print("KEY PLAYER CHECKS:")
key_players = ["Harry Kane", "Leroy Sané", "Kingsley Coman", "Loïs Openda",
               "Victor Boniface", "Benjamin Sesko", "Joshua Kimmich",
               "Granit Xhaka", "Florian Wirtz", "Dayot Upamecano"]
print(df.filter(pl.col("player_name").is_in(key_players))
      .select(["player_name", "role", "season_avg_x", "season_avg_y", "season_avg_y_abs"])
      .sort("season_avg_x", descending=True))

df_save = df.drop(["shot_ratio", "carry_ratio", "duel_ratio",
                   "recovery_ratio", "clearance_ratio", "shot_assist_ratio"])
df_save.write_parquet("../data/processed/player_ratings_FINAL_with_market_values.parquet")
print(" Saved")

KEY PLAYER CHECKS:
shape: (10, 5)
┌─────────────────┬──────┬──────────────┬──────────────┬──────────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y ┆ season_avg_y_abs │
│ ---             ┆ ---  ┆ ---          ┆ ---          ┆ ---              │
│ str             ┆ str  ┆ f64          ┆ f64          ┆ f64              │
╞═════════════════╪══════╪══════════════╪══════════════╪══════════════════╡
│ Kingsley Coman  ┆ LW   ┆ 18.561482    ┆ -3.742539    ┆ 23.581972        │
│ Loïs Openda     ┆ ST   ┆ 17.631121    ┆ 0.181242     ┆ 16.39865         │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6.59159      ┆ 14.136506        │
│ Leroy Sané      ┆ LW   ┆ 16.257619    ┆ -7.700719    ┆ 20.215575        │
│ Harry Kane      ┆ ST   ┆ 15.252045    ┆ 0.509007     ┆ 13.775215        │
│ Florian Wirtz   ┆ CF   ┆ 14.364574    ┆ 4.62817      ┆ 18.115119        │
│ Benjamin Sesko  ┆ CAM  ┆ 9.751801     ┆ 0.861211     ┆ 15.34843         │
│ Joshua Kimmich  ┆ CM   ┆ 3.432657     ┆ -8.760889   

In [39]:
print(df.group_by(["position", "role"]).len()
      .sort(["position", "len"], descending=[False, True]))

shape: (19, 3)
┌────────────┬──────┬─────┐
│ position   ┆ role ┆ len │
│ ---        ┆ ---  ┆ --- │
│ str        ┆ str  ┆ u32 │
╞════════════╪══════╪═════╡
│ defender   ┆ CB   ┆ 73  │
│ defender   ┆ GK   ┆ 21  │
│ defender   ┆ LB   ┆ 4   │
│ defender   ┆ RB   ┆ 3   │
│ forward    ┆ CAM  ┆ 32  │
│ …          ┆ …    ┆ …   │
│ midfielder ┆ CAM  ┆ 22  │
│ midfielder ┆ RB   ┆ 18  │
│ midfielder ┆ LB   ┆ 16  │
│ midfielder ┆ CB   ┆ 8   │
│ midfielder ┆ CDM  ┆ 7   │
└────────────┴──────┴─────┘


In [40]:
def classify_role(avg_x, avg_y, avg_y_abs,
                  shot_r, carry_r, duel_ratio, recovery_r, clearance_r,
                  shot_assist_r, broad_position):
    if avg_x is None or avg_y is None:
        return "Unknown"

    side = "L" if avg_y < 0 else "R"
    pos  = (broad_position or "").lower()

    # ── FORWARDS — only attacking roles ──────────────────────────────────────
    if pos == "forward":
        if avg_y_abs > 20 and carry_r > shot_r * 2.0:
            return f"{side}W"
        elif shot_r > 0.03:
            return "ST"
        else:
            return "CF"

    # ── DEFENDERS — only defensive roles ─────────────────────────────────────
    if pos == "defender":
        if avg_x < -38:
            return "GK"
        if avg_y_abs > 19:
            return f"{side}B"
        return "CB"

    # ── MIDFIELDERS ───────────────────────────────────────────────────────────
    if pos == "midfielder":
        if avg_y_abs > 19:
            return f"{side}M"
        if avg_x >= 8:
            return "CAM"
        elif avg_x >= 2:
            return "CM" if avg_y_abs < 14 else f"{side}M"
        elif avg_x >= -5:
            return "CDM" if (recovery_r > 0.07 or clearance_r > 0.025) else "CM"
        else:
            return "CDM"

    return "CM"

# Verify first
print(classify_role(16.257, -7.700, 20.215, 0.02638, 0.36077, 0.05, 0.04, 0.01, 0.01, "forward"))   # → LW
print(classify_role(15.252,  0.509, 13.775, 0.065,   0.20,    0.06, 0.04, 0.02, 0.008, "forward"))  # → ST
print(classify_role(-10.513, -9.289, 14.600, 0.01,   0.15,    0.07, 0.05, 0.03, 0.005, "defender")) # → CB
print(classify_role(3.432,  -8.760, 17.935, 0.01,    0.25,    0.05, 0.04, 0.01, 0.01, "midfielder"))# → CM or LM

# Apply
roles = []
for row in df.iter_rows(named=True):
    role = classify_role(
        row.get("season_avg_x"),
        row.get("season_avg_y"),
        row.get("season_avg_y_abs") or 0,
        row.get("shot_ratio",        0) or 0,
        row.get("carry_ratio",       0) or 0,
        row.get("duel_ratio",        0) or 0,
        row.get("recovery_ratio",    0) or 0,
        row.get("clearance_ratio",   0) or 0,
        row.get("shot_assist_ratio", 0) or 0,
        row.get("position", ""),
    )
    roles.append(role)

df = df.with_columns(pl.Series("role", roles))

# Check distribution and key players
print("\nROLE DISTRIBUTION:")
print(df.group_by(["position", "role"]).len().sort(["position", "len"], descending=[False, True]))

key_players = ["Harry Kane", "Leroy Sané", "Kingsley Coman", "Loïs Openda",
               "Victor Boniface", "Benjamin Sesko", "Joshua Kimmich",
               "Granit Xhaka", "Florian Wirtz", "Dayot Upamecano"]
print("\nKEY PLAYER CHECKS:")
print(df.filter(pl.col("player_name").is_in(key_players))
      .select(["player_name", "role", "season_avg_x", "season_avg_y"])
      .sort("season_avg_x", descending=True))

# Save
df_save = df.drop(["shot_ratio", "carry_ratio", "duel_ratio",
                   "recovery_ratio", "clearance_ratio", "shot_assist_ratio"])
df_save.write_parquet("../data/processed/player_ratings_FINAL_with_market_values.parquet")
print("\n Saved")

LW
ST
CB
LM

ROLE DISTRIBUTION:
shape: (13, 3)
┌────────────┬──────┬─────┐
│ position   ┆ role ┆ len │
│ ---        ┆ ---  ┆ --- │
│ str        ┆ str  ┆ u32 │
╞════════════╪══════╪═════╡
│ defender   ┆ CB   ┆ 75  │
│ defender   ┆ GK   ┆ 21  │
│ defender   ┆ LB   ┆ 4   │
│ defender   ┆ RB   ┆ 1   │
│ forward    ┆ ST   ┆ 41  │
│ …          ┆ …    ┆ …   │
│ midfielder ┆ LM   ┆ 64  │
│ midfielder ┆ RM   ┆ 63  │
│ midfielder ┆ CDM  ┆ 41  │
│ midfielder ┆ CAM  ┆ 20  │
│ midfielder ┆ CM   ┆ 19  │
└────────────┴──────┴─────┘

KEY PLAYER CHECKS:
shape: (10, 4)
┌─────────────────┬──────┬──────────────┬──────────────┐
│ player_name     ┆ role ┆ season_avg_x ┆ season_avg_y │
│ ---             ┆ ---  ┆ ---          ┆ ---          │
│ str             ┆ str  ┆ f64          ┆ f64          │
╞═════════════════╪══════╪══════════════╪══════════════╡
│ Kingsley Coman  ┆ LW   ┆ 18.561482    ┆ -3.742539    │
│ Loïs Openda     ┆ ST   ┆ 17.631121    ┆ 0.181242     │
│ Victor Boniface ┆ ST   ┆ 17.23027     ┆ 6